# Topic Modeling of Policy-Cited Literature

This notebook performs topic modeling on the cleaned publication
datasets produced by the data-preparation workflow.

The analysis includes text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, and temporal analysis.

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile

# Project directories
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("Output directory  :", OUTPUT_DIR)

Project directory : /nfs/mfirdausi/project/review_paper_2
Data directory    : /nfs/mfirdausi/project/review_paper_2/data
Output directory  : /nfs/mfirdausi/project/review_paper_2/output


## 1. Load Cleaned Publication Data

The cleaned Overton and Scopus publication datasets produced by the
data-preparation notebook are loaded separately. The two sources are
retained as distinct datasets rather than merged.

In [17]:
OVERTON_FILE = (
    DATA_DIR / "overton_full_clean.xlsx"
)

SCOPUS_FILE = (
    DATA_DIR / "scopus_full_clean.xlsx"
)

overton = pd.read_excel(
    OVERTON_FILE
)

scopus = pd.read_excel(
    SCOPUS_FILE
)

print("Overton")
print("-------")
print(f"Documents : {len(overton):,}")
print(f"Columns   : {len(overton.columns):,}")

print("\nScopus")
print("------")
print(f"Documents : {len(scopus):,}")
print(f"Columns   : {len(scopus.columns):,}")

Overton
-------
Documents : 14,266
Columns   : 21

Scopus
------
Documents : 16,403
Columns   : 18


### 1.1 Define the Independent Topic-Modeling Corpora

The Overton and Scopus publication collections are analyzed as
independent corpora. Each corpus therefore receives its own text
preprocessing, document-term matrix, LDA model-selection procedure,
final topic model, and downstream topic analysis.

In [19]:
corpora = {
    "overton": overton.copy(),
    "scopus": scopus.copy(),
}

for corpus_name, corpus_df in corpora.items():

    print(f"{corpus_name.upper()}")
    print("-" * len(corpus_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print()

OVERTON
-------
Documents          : 14,266
Non-missing titles : 14,266
Non-missing abstracts: 14,264
Missing abstracts  : 2

SCOPUS
------
Documents          : 16,403
Non-missing titles : 16,403
Non-missing abstracts: 16,403
Missing abstracts  : 0



## 2. Prepare Corpora for Topic Modeling

Topic modeling is performed independently for the Overton and Scopus
datasets. Publications without abstracts are excluded because the LDA
models are estimated from abstract text.

In [20]:
lda_corpora = {}

for corpus_name, corpus_df in corpora.items():

    lda_df = (
        corpus_df[
            corpus_df["Abstract"].notna()
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Remove abstracts that are empty after whitespace stripping.
    lda_df["Abstract"] = (
        lda_df["Abstract"]
        .astype(str)
        .str.strip()
    )

    lda_df = (
        lda_df[
            lda_df["Abstract"] != ""
        ]
        .reset_index(drop=True)
    )

    lda_corpora[corpus_name] = lda_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))
    print(
        f"Input publications : "
        f"{len(corpus_df):,}"
    )
    print(
        f"LDA documents      : "
        f"{len(lda_df):,}"
    )
    print(
        f"Excluded           : "
        f"{len(corpus_df) - len(lda_df):,}"
    )
    print()
    

OVERTON
-------
Input publications : 14,266
LDA documents      : 14,264
Excluded           : 2

SCOPUS
------
Input publications : 16,403
LDA documents      : 16,403
Excluded           : 0



### 2.1 Corpus Relevance Diagnostic

Before topic modeling, the retrieved publications are screened
diagnostically for terminology associated with electrical power and
energy systems. This step evaluates whether the search results contain
substantial off-domain literature that could distort the latent topic
structure.

In [21]:
POWER_DOMAIN_TERMS = [
    r"\bpower system",
    r"\bpower systems",
    r"\belectric power",
    r"\belectrical power",
    r"\bpower grid",
    r"\belectric grid",
    r"\belectrical grid",
    r"\bsmart grid",
    r"\bmicrogrid",
    r"\bmicro-grid",
    r"\btransmission system",
    r"\bdistribution system",
    r"\bpower network",
    r"\belectricity",
    r"\bvoltage",
    r"\breactive power",
    r"\bactive power",
    r"\boptimal power flow",
    r"\bload flow",
    r"\bpower flow",
]

power_pattern = re.compile(
    "|".join(POWER_DOMAIN_TERMS),
    flags=re.IGNORECASE,
)

for corpus_name, corpus_df in lda_corpora.items():

    relevant_mask = (
        corpus_df["Title"]
        .fillna("")
        .str.contains(
            power_pattern,
            regex=True,
        )
        |
        corpus_df["Abstract"]
        .fillna("")
        .str.contains(
            power_pattern,
            regex=True,
        )
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents             : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Power-domain terminology    : "
        f"{relevant_mask.sum():,}"
    )

    print(
        f"No power-domain terminology : "
        f"{(~relevant_mask).sum():,}"
    )

    print(
        f"Share with domain terminology: "
        f"{relevant_mask.mean() * 100:.2f}%"
    )

    print()

OVERTON
-------
Total documents             : 14,264
Power-domain terminology    : 3,515
No power-domain terminology : 10,749
Share with domain terminology: 24.64%

SCOPUS
------
Total documents             : 16,403
Power-domain terminology    : 10,234
No power-domain terminology : 6,169
Share with domain terminology: 62.39%



In [22]:
relevance_diagnostics = {}

for corpus_name, corpus_df in lda_corpora.items():

    text = (
        corpus_df["Title"].fillna("")
        + " "
        + corpus_df["Abstract"].fillna("")
    )

    relevant_mask = text.str.contains(
        power_pattern,
        regex=True,
    )

    diagnostic_df = (
        corpus_df.loc[
            ~relevant_mask,
            [
                "Title",
                "Year",
                "DOI",
                "Abstract",
            ],
        ]
        .copy()
    )

    relevance_diagnostics[
        corpus_name
    ] = diagnostic_df

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        diagnostic_df[
            [
                "Title",
                "Year",
                "DOI",
            ]
        ]
        .sample(
            n=min(
                30,
                len(diagnostic_df),
            ),
            random_state=123,
        )
        .reset_index(drop=True)
    )


OVERTON
-------


,Title,Year,DOI
0,Development and mean life of aluminum first-su...,2009,10.1016/j.solmat.2009.05.004
1,High Reliability Safeguards approach to remote...,2017,10.1016/j.nucengdes.2017.08.012
2,Economic valuation of coccidioidomycosis (vall...,2021,10.1175/WCAS-D-20-0036.1
3,Taxi time prediction at Charlotte airport usin...,2015,10.2514/6.2015-2272
4,Urban Development and Energy Access in Informa...,2016,10.1016/j.proeng.2016.08.680
5,Model order reduction a key technology for dig...,2018,10.1007/978-3-319-75319-5_8
6,Superior room-temperature ductility of typical...,2016,10.1038/ncomms12261
7,Fourier-bessel series model for the Stefan pro...,2018,10.1115/ICONE26-81009
8,A simple apparatus for the injection of lithiu...,2010,10.1016/j.fusengdes.2010.08.033
9,The 'southern model' of welfare in social Europe,1996,10.1177/095892879600600102



SCOPUS
------


,Title,Year,DOI
0,Power optimisation scheme of induction motor u...,2020,10.1049/iet-est.2019.0151
1,An Adaptive Sparse Anisotropic Polynomial-Chao...,2021,10.1109/MEMC.2021.9477248
2,Artificial intelligence in renewable energy te...,2026,10.1016/j.nxener.2026.100575
3,High-performance computing for electric vehicl...,2024,10.4018/978-1-6684-3795-7.ch016
4,Smart energy efficient transportation systems,2026,10.1016/B978-0-323-95045-9.00023-8
5,NEAR-OPTIMAL SOLUTIONS OF CONSTRAINED LEARNING...,2024,0
6,Asynchronous observer-based control for input-...,2026,10.1016/j.cnsns.2025.109458
7,Multi-Objective Optimal Design of an On-Grid H...,2026,10.1109/ACCESS.2026.3667538
8,A review of distributed energy system optimiza...,2023,10.1016/j.jobe.2023.106735
9,Advances in Mountain Gazelle Optimizer: A Comp...,2025,10.1007/s44196-025-00968-4


### 2.2 Domain-Relevance Screening

To support a meaningful comparison between policy-facing and academic
literature, both corpora are restricted to publications relevant to
the power and energy systems domain before topic modeling.

The relevance screen is applied identically to Overton and Scopus.
A broad domain vocabulary is used to capture power-system research
without requiring specific terminology such as "power system" or
"power flow". The screening rule is validated through manual
inspection of retained and excluded records before the filtered
corpora are used for LDA.

In [23]:
# Broad power-and-energy-system terminology used for relevance screening.
#
# This is intentionally broader than the earlier diagnostic.
# The same rule is applied to both Overton and Scopus.

DOMAIN_TERM_GROUPS = {

    "power_system": [
        r"\bpower systems?\b",
        r"\belectric(?:al)? power\b",
        r"\bpower networks?\b",
        r"\belectric(?:al)? networks?\b",
        r"\bpower grids?\b",
        r"\belectric(?:al)? grids?\b",
        r"\bsmart grids?\b",
        r"\bmicrogrids?\b",
        r"\bmicro-grids?\b",
    ],

    "power_flow_operation": [
    r"\bpower flows?\b",
    r"\bload flows?\b",
    r"\boptimal power flows?\b",
    r"\bopf\b",

    # Dispatch and system operation
    r"\beconomic power dispatch\b",
    r"\beconomic dispatch\b",
    r"\boptimal dispatch\b",
    r"\bpower dispatch\b",
    r"\bunit commitment\b",

    # Loss / operating quantities
    r"\bpower loss(?:es)?\b",
    r"\breal power\b",
    r"\bactive power\b",
    r"\breactive power\b",

    # State / voltage / frequency operation
    r"\bstate estimation\b",
    r"\bvoltage stability\b",
    r"\bvoltage control\b",
    r"\bfrequency control\b",
    r"\bfrequency regulation\b",
    ],

    "transmission_distribution": [
        r"\btransmission systems?\b",
        r"\btransmission networks?\b",
        r"\bdistribution systems?\b",
        r"\bdistribution networks?\b",
        r"\bdistribution grids?\b",
        r"\btransmission grids?\b",
        r"\bdistribution feeders?\b",
        r"\bfeeders?\b",
        r"\bsubstations?\b",
    ],

    "electricity": [
        r"\belectricity\b",
        r"\belectric energy\b",
        r"\belectrical energy\b",
        r"\belectricity markets?\b",
        r"\benergy markets?\b",
        r"\belectric utilities?\b",
        r"\bpower utilities?\b",
    ],

    "generation_resources": [
    r"\bpower generation\b",
    r"\belectricity generation\b",
    r"\bgenerating units?\b",
    r"\bgenerators?\b",
    r"\bdistributed generation\b",
    r"\bdistributed energy resources?\b",
    r"\bder\b",
    r"\bders\b",
    r"\benergy resources?\b",
    ],

    "renewables": [
        r"\brenewable energy\b",
        r"\brenewable generation\b",
        r"\bsolar energy\b",
        r"\bsolar power\b",
        r"\bphotovoltaic\b",
        r"\bphotovoltaics\b",
        r"\bpv systems?\b",
        r"\bwind energy\b",
        r"\bwind power\b",
        r"\bwind farms?\b",
        r"\bwind turbines?\b",
    ],

    "storage_ev": [
        r"\benergy storage\b",
        r"\bbattery storage\b",
        r"\bbattery energy storage\b",
        r"\bbess\b",
        r"\belectric vehicles?\b",
        r"\bev charging\b",
        r"\bvehicle-to-grid\b",
        r"\bv2g\b",
    ],

    "power_electronics": [
        r"\bpower electronics\b",
        r"\binverters?\b",
        r"\bconverters?\b",
        r"\bac[- ]dc\b",
        r"\bdc[- ]ac\b",
    ],

    "load_demand": [
        r"\belectric(?:al)? loads?\b",
        r"\bload demand\b",
        r"\belectricity demand\b",
        r"\benergy demand\b",
        r"\bload forecasting\b",
        r"\bdemand response\b",
        r"\bdemand-side management\b",
    ],

    "energy_system": [
    r"\benergy systems?\b",
    r"\bintegrated energy systems?\b",
    r"\bmulti-energy systems?\b",
    r"\bmultienergy systems?\b",
    r"\benergy management systems?\b",
    r"\benergy management\b",
    ],

    "market_reliability": [
    r"\blocational marginal pric(?:e|es|ing)\b",
    r"\blmp\b",
    r"\belectricity pric(?:e|es|ing)\b",
    r"\benergy pric(?:e|es|ing)\b",
    r"\bpower system reliability\b",
    r"\bgrid reliability\b",
    r"\bline failures?\b",
    r"\btransmission line failures?\b",
    r"\bpower system security\b",
    r"\bgrid security\b",
    ],
}

In [24]:
domain_patterns = {
    group: re.compile(
        "|".join(patterns),
        flags=re.IGNORECASE,
    )
    for group, patterns in DOMAIN_TERM_GROUPS.items()
}

domain_screening = {}

for corpus_name, corpus_df in lda_corpora.items():

    screening_df = corpus_df.copy()

    screening_text = (
        screening_df["Title"]
        .fillna("")
        .astype(str)
        + " "
        + screening_df["Abstract"]
        .fillna("")
        .astype(str)
    )

    matched_columns = []

    for group, pattern in domain_patterns.items():

        column = f"match_{group}"

        screening_df[column] = (
            screening_text.str.contains(
                pattern,
                regex=True,
            )
        )

        matched_columns.append(column)

    # Number of different domain concept groups matched.
    screening_df[
        "Domain_Group_Count"
    ] = (
        screening_df[
            matched_columns
        ]
        .sum(axis=1)
    )

    # Initial broad relevance rule:
    # at least one substantive power/energy-domain group.
    screening_df[
        "Domain_Relevant"
    ] = (
        screening_df[
            "Domain_Group_Count"
        ] >= 1
    )

    domain_screening[
        corpus_name
    ] = screening_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents       : "
        f"{len(screening_df):,}"
    )

    print(
        f"Domain relevant       : "
        f"{screening_df['Domain_Relevant'].sum():,}"
    )

    print(
        f"Not domain relevant   : "
        f"{(~screening_df['Domain_Relevant']).sum():,}"
    )

    print(
        f"Retention rate        : "
        f"{screening_df['Domain_Relevant'].mean() * 100:.2f}%"
    )

    print()

OVERTON
-------
Total documents       : 14,264
Domain relevant       : 5,045
Not domain relevant   : 9,219
Retention rate        : 35.37%

SCOPUS
------
Total documents       : 16,403
Domain relevant       : 12,042
Not domain relevant   : 4,361
Retention rate        : 73.41%



In [28]:
domain_group_summary = []

for corpus_name, screening_df in domain_screening.items():

    for group in DOMAIN_TERM_GROUPS:

        count = int(
            screening_df[
                f"match_{group}"
            ].sum()
        )

        domain_group_summary.append({
            "Corpus": corpus_name.capitalize(),
            "Domain_Group": group,
            "Documents": count,
            "Percent": (
                count
                / len(screening_df)
                * 100
            ),
        })

domain_group_summary = pd.DataFrame(
    domain_group_summary
)

domain_group_summary.pivot(
    index="Domain_Group",
    columns="Corpus",
    values="Documents",
)

Corpus,Overton,Scopus
Domain_Group,,
electricity,1366,2035
energy_system,564,2390
generation_resources,1282,3485
load_demand,427,1362
market_reliability,177,546
power_electronics,475,1140
power_flow_operation,917,5747
power_system,1989,7107
renewables,2139,4603


In [29]:
for corpus_name, screening_df in domain_screening.items():

    print(f"\n{'=' * 70}")
    print(corpus_name.upper())
    print(f"{'=' * 70}")

    retained = screening_df[
        screening_df["Domain_Relevant"]
    ]

    rejected = screening_df[
        ~screening_df["Domain_Relevant"]
    ]

    print("\nRANDOM RETAINED DOCUMENTS")
    print("-------------------------")

    display(
        retained[
            [
                "Title",
                "Year",
                "DOI",
                "Domain_Group_Count",
            ]
        ]
        .sample(
            n=min(20, len(retained)),
            random_state=123,
        )
        .reset_index(drop=True)
    )

    print("\nRANDOM REJECTED DOCUMENTS")
    print("-------------------------")

    display(
        rejected[
            [
                "Title",
                "Year",
                "DOI",
            ]
        ]
        .sample(
            n=min(20, len(rejected)),
            random_state=123,
        )
        .reset_index(drop=True)
    )


OVERTON

RANDOM RETAINED DOCUMENTS
-------------------------


,Title,Year,DOI,Domain_Group_Count
0,Forecasting day-ahead price of electricity - A...,2013,10.1504/IJBEX.2013.056110,2
1,Exploring electric vehicle charging patterns: ...,2020,10.1016/j.trd.2020.102249,1
2,Condition monitoring of wind turbines: Techniq...,2012,10.1016/j.renene.2012.03.003,1
3,Validation of combined analytical methods to p...,2020,10.1016/j.triboint.2020.106347,1
4,Distributed energy resources and the organized...,2019,10.1016/j.enpol.2018.11.009,3
5,Reduced-order structure-preserving model for p...,2017,10.1109/COMPEL.2017.8013389,2
6,AI-assistance for predictive maintenance of re...,2021,10.1016/j.energy.2021.119775,2
7,Agent-based control framework for distributed ...,2006,10.1109/IAT.2006.27,6
8,Optimization methods applied to renewable and ...,2011,10.1016/j.rser.2010.12.008,2
9,A Setting-Free Differential Protection for Pow...,2019,10.1109/TPWRD.2018.2889471,1



RANDOM REJECTED DOCUMENTS
-------------------------


,Title,Year,DOI
0,The economic burden of physical inactivity: a ...,2016,10.1016/S0140-6736(16)30383-X
1,Challenges to Transforming Unconventional Soci...,2020,10.1017/dmp.2019.92
2,Water transport mechanisms for salt-rejecting ...,2018,10.1016/j.memsci.2018.05.041
3,Associations of mortality with long-term expos...,2015,10.1289/ehp.1408565
4,The impact of synchronisation on secure inform...,2001,10.1007/3-540-45575-2_22
5,Options for reforming agricultural subsidies f...,2022,10.1038/s41467-021-27645-2
6,Distributed snapshots for mobile computing sys...,2004,10.1109/PERCOM.2004.1276856
7,Handling SQL Databases in Automated System Tes...,2020,10.1145/3391533
8,CCured: Type-safe retrofitting of legacy software,2005,10.1145/1065887.1065892
9,Geometry of Kapitsa's potentials,1998,10.1088/0951-7715/11/5/011



SCOPUS

RANDOM RETAINED DOCUMENTS
-------------------------


,Title,Year,DOI,Domain_Group_Count
0,Power systems and microgrids resilience enhanc...,2025,10.1016/j.rser.2024.114953,1
1,Transfer Learning-Based Model Training for Sho...,2026,10.35833/MPCE.2024.000940,3
2,Machine-learned security assessment for changi...,2022,10.1016/j.ijepes.2021.107380,2
3,Robustness of Evolving Power Grids: Modeling a...,2026,10.1201/9781003622130,2
4,Thyristor controlled series compensator planni...,2010,10.1109/PECON.2010.5697550,3
5,Optimal design and operation of a power-to-gas...,2026,10.1016/j.egyr.2026.109680,4
6,Scalable Optimization Methods for Distribution...,2016,10.1109/TSG.2016.2543264,4
7,Probabilistic Power Flow Method for Hybrid AC/...,2023,10.3390/en16062547,4
8,Genetic search for an optimal power flow solut...,2008,0,1
9,Convex Optimization of Power Systems,2015,10.1017/9781139924672,2



RANDOM REJECTED DOCUMENTS
-------------------------


,Title,Year,DOI
0,A Review of Optimal Design for Large-Scale Mic...,2023,10.3390/agronomy13122966
1,Integration of Robust Control and Multi-Object...,2025,10.1002/oca.3249
2,Accuracy analysis of rebar quantity estimation...,2026,10.1080/15623599.2026.2707233
3,Convergence Track Based Adaptive Differential ...,2022,10.32604/cmc.2022.024211
4,Nondestructive detection of trace cadmium in l...,2025,10.1016/j.jfca.2025.108038
5,Survey on Lagrangian relaxation for MILP: impo...,2024,10.1007/s10479-023-05499-9
6,The Comprehensive Review for Biobased FPA Algo...,2021,10.1002/9781119681984.ch7
7,Data-Driven Optimal Scheduling Algorithm of Hu...,2022,10.1155/2022/8602015
8,A modified teaching–learning-based optimizatio...,2019,10.1007/s13042-018-0815-8
9,Dual mutations collaboration mechanism with el...,2022,10.1007/s00500-021-06454-1


### 2.3 Final Domain-Filtered Corpora

In [30]:
lda_corpora_filtered = {}

for corpus_name, screening_df in domain_screening.items():

    filtered_df = (
        screening_df[
            screening_df["Domain_Relevant"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    lda_corpora_filtered[
        corpus_name
    ] = filtered_df

    original_n = len(
        lda_corpora[corpus_name]
    )

    filtered_n = len(
        filtered_df
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Original documents : "
        f"{original_n:,}"
    )

    print(
        f"Retained documents : "
        f"{filtered_n:,}"
    )

    print(
        f"Excluded documents : "
        f"{original_n - filtered_n:,}"
    )

    print(
        f"Retention rate     : "
        f"{filtered_n / original_n * 100:.2f}%"
    )

    print()

OVERTON
-------
Original documents : 14,264
Retained documents : 5,045
Excluded documents : 9,219
Retention rate     : 35.37%

SCOPUS
------
Original documents : 16,403
Retained documents : 12,042
Excluded documents : 4,361
Retention rate     : 73.41%



## 3. R Text-Processing Backend

The abstract corpora are preprocessed using R `tm` and `SnowballC`
through `Rscript`. This preserves the text-processing methodology used
for the LDA analysis while allowing the complete workflow to be
controlled from Python.

In [ ]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"/nfs/mfirdausi/miniconda3/envs/pytorch/bin/Rscript"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

### 3.1 Text-Preprocessing Configuration

The Overton and Scopus corpora are processed using an identical text
preprocessing configuration to support direct comparison between the
two independently estimated topic models.

Standard English stopwords are supplemented with terms appearing
explicitly in the literature-search query because these terms define
the corpus but provide limited information for distinguishing latent
topics.

In [ ]:
# Search-query terms removed from both corpora.
# change this based on your topic

QUERY_STOPWORDS = [
    # Search-query terms
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",

    # Publisher/copyright boilerplate
    "©",
]

# Initial DTM configuration.
MIN_TERM_LENGTH = 3
MIN_DOC_FREQ = 3

print("Query-specific stopwords:")
for word in QUERY_STOPWORDS:
    print(f"  - {word}")

print("\nDTM configuration")
print("-----------------")
print("Minimum term length     :", MIN_TERM_LENGTH)
print("Minimum document freq.  :", MIN_DOC_FREQ)

### 3.2 Preprocess Abstracts with R `tm`

The same R `tm` and `SnowballC` preprocessing pipeline is applied
independently to the Overton and Scopus abstracts. Processing includes
lowercasing, punctuation and number removal, whitespace normalization,
English and query-specific stopword removal, and English Snowball
stemming.

The resulting corpora are used to inspect vocabulary characteristics
before the final document-term matrices are constructed.

In [ ]:
R_PREPROCESS_DIR = OUTPUT_DIR / "r_preprocessing"

R_PREPROCESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR / "preprocess_corpus.R"
)

r_preprocess_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Load abstracts
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

abstracts <- data$Abstract

# ------------------------------------------------------------
# Shared stopwords
# ------------------------------------------------------------

custom_stops <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",
    "©"
)

all_stops <- unique(
    c(
        tm::stopwords("english"),
        custom_stops
    )
)

# ------------------------------------------------------------
# tm preprocessing
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(abstracts)
)

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    all_stops
)

# Remove copyright symbol explicitly.
corpus <- tm_map(
    corpus,
    content_transformer(
        function(x) gsub(
            "©",
            " ",
            x,
            fixed = TRUE
        )
    )
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

processed <- vapply(
    corpus,
    as.character,
    character(1)
)

result <- data.frame(
    document_id = seq_along(processed),
    processed_text = processed,
    stringsAsFactors = FALSE
)

write.csv(
    result,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created R preprocessing script:",
    R_PREPROCESS_SCRIPT
)

In [ ]:
processed_corpora = {}

for corpus_name, corpus_df in lda_corpora_filtered.items():

    input_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )
    # ---------------------------------------------------------
    # Use cached processed corpus if it already exists
    # ---------------------------------------------------------

    if output_file.exists():

        print(
            f"Loading cached {corpus_name.upper()} "
            f"processed corpus ..."
        )

    else:

        print(
            f"Processing {corpus_name.upper()} with R ..."
        )

        # Export abstracts only when preprocessing is required.
        corpus_df[
            ["Abstract"]
        ].to_csv(
            input_file,
            index=False,
            encoding="utf-8",
        )

        run = subprocess.run(
            [
                str(RSCRIPT),
                str(R_PREPROCESS_SCRIPT),
                str(input_file),
                str(output_file),
            ],
            capture_output=True,
            text=True,
            check=True,
        )

    # ---------------------------------------------------------
    # Load processed corpus
    # ---------------------------------------------------------

    processed_df = pd.read_csv(
        output_file,
        keep_default_na=False,
    )

    processed_corpora[
        corpus_name
    ] = processed_df

    print(
        f"Documents processed : "
        f"{len(processed_df):,}"
    )

    print(
        f"Empty documents     : "
        f"{(processed_df['processed_text'].str.strip() == '').sum():,}"
    )

    print()

In [ ]:
for corpus_name, processed_df in processed_corpora.items():

    token_counts = (
        processed_df[
            "processed_text"
        ]
        .str.split()
        .str.len()
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{len(processed_df):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(token_counts.sum()):,}"
    )

    print(
        f"Minimum tokens  : "
        f"{int(token_counts.min()):,}"
    )

    print(
        f"Median tokens   : "
        f"{token_counts.median():.0f}"
    )

    print(
        f"Mean tokens     : "
        f"{token_counts.mean():.1f}"
    )

    print(
        f"Maximum tokens  : "
        f"{int(token_counts.max()):,}"
    )

    print()

### 3.3 Inspect Frequent Terms

The most frequent terms remaining after preprocessing are inspected
before constructing the final document-term matrices. This diagnostic
is used to identify high-frequency generic terms that provide little
thematic discrimination and may therefore warrant inclusion in the
shared custom stopword list.

In [ ]:
from collections import Counter

term_frequency_tables = {}

for corpus_name, processed_df in processed_corpora.items():

    term_frequency = Counter(
        token
        for text in processed_df["processed_text"]
        for token in text.split()
    )

    top_terms = pd.DataFrame(
        term_frequency.most_common(50),
        columns=[
            "term",
            "frequency",
        ],
    )

    term_frequency_tables[
        corpus_name
    ] = top_terms

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        top_terms.head(30)
    )

## 4. Document-Term Matrix Construction

Separate document-term matrices are constructed for the Overton and
Scopus corpora using R `tm`. The same preprocessing and vocabulary
filtering rules are applied to both corpora.

Terms shorter than three characters and terms occurring in fewer than
three documents are excluded. The resulting matrix dimensions and
sparsity are inspected before LDA model selection.

In [ ]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- 3

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)

In [ ]:
dtm_summaries = {}
dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    # Domain-filtered processed corpus
    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )

    # Keep domain-filtered DTM diagnostics separate
    # from the previous unfiltered analysis.
    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    dtm_summaries[
        corpus_name
    ] = summary_df

    dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Vocabulary      : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density  : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

### 4.1 Vocabulary Filtering

In [ ]:
# Compare vocabulary sizes under alternative document-frequency
# thresholds using the document frequencies already calculated in R.

DF_THRESHOLDS = [
    3,
    5,
    10,
    15,
    20,
    25,
    50,
    100,
]

df_threshold_results = []

for corpus_name, terms_df in dtm_term_tables.items():

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        df_threshold_results.append({
            "Corpus": corpus_name.capitalize(),
            "Min_Document_Frequency": threshold,
            "Retained_Terms": int(retained),
        })

df_threshold_results = pd.DataFrame(
    df_threshold_results
)

df_threshold_pivot = (
    df_threshold_results
    .pivot(
        index="Min_Document_Frequency",
        columns="Corpus",
        values="Retained_Terms",
    )
)

df_threshold_pivot

In [ ]:
for corpus_name, terms_df in dtm_term_tables.items():

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    n_docs = int(
        dtm_summaries[
            corpus_name
        ].iloc[0]["Documents"]
    )

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        print(
            f"DF >= {threshold:3d} "
            f"({threshold / n_docs * 100:5.3f}% docs)"
            f" : {retained:6,d} terms"
        )

    print()

Based on the threshold analysis, the final vocabulary filter is defined
proportionally rather than using a common absolute document-frequency
count. Terms must occur in at least 0.2% of documents within each
corpus.

This corresponds to a minimum document frequency of 11 documents for
Overton and 25 documents for Scopus. The proportional rule provides
comparable vocabulary filtering despite the different corpus sizes.

### 4.2 Construct the Final Document-Term Matrices

The final document-term matrices are constructed using a minimum term
length of three characters and a minimum document frequency equal to
0.2% of the documents in each corpus.

In [ ]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]
min_doc_freq <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)


In [ ]:
import math

MIN_DOC_PERCENT = 0.002

final_dtm_summaries = {}
final_dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )

    n_documents = len(
        processed_corpora[corpus_name]
    )

    min_doc_freq = math.ceil(
        n_documents * MIN_DOC_PERCENT
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
            str(min_doc_freq),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    final_dtm_summaries[
        corpus_name
    ] = summary_df

    final_dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents            : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Minimum document freq: "
        f"{min_doc_freq:,}"
    )

    print(
        f"DF threshold         : "
        f"{min_doc_freq / n_documents * 100:.3f}%"
    )

    print(
        f"Vocabulary           : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries      : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens         : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density       : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

## 5. LDA Topic Modeling and Model Selection

LDA models are estimated independently for the domain-filtered Overton
and Scopus corpora using R `topicmodels` with Gibbs sampling.

For model selection, each corpus is divided reproducibly into 80%
training and 20% held-out documents using a fixed random seed.
Candidate topic numbers are evaluated using held-out predictive
performance together with topic coherence and distinctiveness.

In [ ]:
LDA_DIR = OUTPUT_DIR / "lda"

LDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("LDA output directory:")
print(LDA_DIR)

### 5.1 Training and Held-Out Splits

An independent 80/20 split is generated for each corpus using R's
random-number generator with seed 123. The same sampling procedure is
therefore applied to both datasets.

In [ ]:
RANDOM_SEED = 123

lda_splits = {}

for corpus_name, summary_df in final_dtm_summaries.items():

    n_documents = int(
        summary_df.iloc[0]["Documents"]
    )

    split_result = subprocess.run(
        [
            str(RSCRIPT),
            "-e",
            (
                f"set.seed({RANDOM_SEED}); "
                f"n <- {n_documents}; "
                "train_id <- sample("
                "seq_len(n), "
                "size=floor(0.8*n), "
                "replace=FALSE"
                "); "
                'cat(train_id, sep=",")'
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    # Keep R's 1-based indices because these will
    # subsequently be passed back to R topicmodels.
    train_id = np.fromstring(
        split_result.stdout.strip(),
        sep=",",
        dtype=int,
    )

    all_id = np.arange(
        1,
        n_documents + 1
    )

    test_id = np.setdiff1d(
        all_id,
        train_id
    )

    lda_splits[corpus_name] = {
        "train_id": train_id,
        "test_id": test_id,
    }

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents    : "
        f"{n_documents:,}"
    )

    print(
        f"Training documents : "
        f"{len(train_id):,}"
    )

    print(
        f"Held-out documents : "
        f"{len(test_id):,}"
    )

    print(
        f"Overlap             : "
        f"{len(np.intersect1d(train_id, test_id))}"
    )

    print()

### 5.2 Coarse Topic-Number Search

A coarse search is performed to identify a plausible range for the
number of latent topics in each domain-filtered corpus. Short Gibbs
chains are used at this screening stage to limit computational cost.

The final corpus-specific vocabulary thresholds established in Section
4 are retained throughout model selection.

In [ ]:
K_COARSE = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

COARSE_BURN_IN = 50
COARSE_ITERATIONS = 100
COARSE_THIN = 10

MIN_DOC_PERCENT = 0.002

print("Coarse K values :", K_COARSE)
print("Burn-in         :", COARSE_BURN_IN)
print("Iterations      :", COARSE_ITERATIONS)
print("Thin            :", COARSE_THIN)

In [ ]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR / "domain_filtered_coarse_k_search.R"
)

r_coarse_search_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_file    <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    5, 10, 15, 20,
    25, 30, 40, 50
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10

# ------------------------------------------------------------
# Load domain-filtered processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Reconstruct FINAL DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ------------------------------------------------------------
# Train / held-out split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Retain terms represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove zero-token documents after training-vocabulary filtering.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(!train_nonempty)
n_empty_test <- sum(!test_nonempty)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Coarse K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Elapsed_seconds = elapsed
        )
    )

    # Preserve completed K values.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

In [ ]:
import math

corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

coarse_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_coarse_k_search.csv"
)

# Same proportional vocabulary rule used in Section 4.
n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print("Running domain-filtered Overton coarse K search...")
print("Documents       :", f"{n_documents:,}")
print("Minimum DF      :", min_doc_freq)
print("K values        :", K_COARSE)

coarse_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_COARSE_SEARCH_SCRIPT),
        str(processed_file),
        str(train_file),
        str(coarse_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    coarse_run.returncode
)

if coarse_run.stdout.strip():
    print("\nR output:")
    print(
        coarse_run.stdout
    )

if coarse_run.stderr.strip():
    print("\nR messages:")
    print(
        coarse_run.stderr
    )

# Load completed results.
if coarse_output.exists():

    overton_coarse_results = pd.read_csv(
        coarse_output
    )

    print(
        "\nCompleted K values:",
        overton_coarse_results["K"].tolist()
    )

    display(
        overton_coarse_results
    )

else:

    print(
        "\nNo completed K values were saved."
    )

#### 5.2.1 Extended Overton Search

Because held-out perplexity continued to decrease through \(K=50\),
the screening range is extended to \(K=60,70,80,100\). These models
retain the same short Gibbs-chain settings and are used only to
characterize the model-complexity trend.

In [ ]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR / "overton_domain_filtered_extended_k_search.R"
)

r_coarse_search_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_file    <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    60, 70, 80, 100
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10

# ------------------------------------------------------------
# Load domain-filtered processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Reconstruct FINAL DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ------------------------------------------------------------
# Train / held-out split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Retain terms represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove zero-token documents after training-vocabulary filtering.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(!train_nonempty)
n_empty_test <- sum(!test_nonempty)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Coarse K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Elapsed_seconds = elapsed
        )
    )

    # Preserve completed K values.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

In [ ]:
import math

K_EXTENDED = [
    60,
    70,
    80,
    100,
]

corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

# IMPORTANT:
# Save separately from the original K=5,...,50 coarse search.
extended_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_extended_k_search.csv"
)

# Same proportional vocabulary rule used in Section 4.
n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Overton extended K search..."
)

print(
    "Documents       :",
    f"{n_documents:,}"
)

print(
    "Minimum DF      :",
    min_doc_freq
)

print(
    "K values        :",
    K_EXTENDED
)

extended_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_COARSE_SEARCH_SCRIPT),
        str(processed_file),
        str(train_file),
        str(extended_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    extended_run.returncode
)

if extended_run.stdout.strip():

    print("\nR output:")

    print(
        extended_run.stdout
    )

if extended_run.stderr.strip():

    print("\nR messages:")

    print(
        extended_run.stderr
    )

# Load completed extended-search results.
if extended_output.exists():

    overton_extended_results = pd.read_csv(
        extended_output
    )

    print(
        "\nCompleted K values:",
        overton_extended_results[
            "K"
        ].tolist()
    )

    display(
        overton_extended_results
    )

else:

    print(
        "\nNo completed extended K values were saved."
    )

In [ ]:
overton_k_screen = (
    pd.concat(
        [
            overton_coarse_results,
            overton_extended_results,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset="K"
    )
    .sort_values("K")
    .reset_index(drop=True)
)

display(overton_k_screen)

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    overton_k_screen["K"],
    overton_k_screen["Perplexity"],
    marker="o",
)

ax.set_xlabel(
    "Number of Topics (K)"
)

ax.set_ylabel(
    "Held-Out Perplexity"
)

ax.set_title(
    "Overton LDA Topic-Number Screening"
)

ax.grid(
    alpha=0.3
)

plt.tight_layout()
plt.show()

### 5.3 Multi-Criterion Candidate Evaluation

Because held-out perplexity continued to decrease throughout the
screening range, perplexity alone does not provide a finite optimum
for the number of topics.

Candidate models at \(K=30,50,70,\) and \(100\) are therefore
re-estimated using longer Gibbs chains and evaluated using held-out
perplexity, semantic coherence, topic distinctiveness, and thematic
interpretability.

In [ ]:
K_CANDIDATES = [
    30,
    50,
    70,
    100,
]

CANDIDATE_BURN_IN = 500
CANDIDATE_ITERATIONS = 1000
CANDIDATE_THIN = 50

TOP_N_COHERENCE = 10

print("Candidate K values :", K_CANDIDATES)
print("Burn-in           :", CANDIDATE_BURN_IN)
print("Iterations        :", CANDIDATE_ITERATIONS)
print("Thin              :", CANDIDATE_THIN)
print("Coherence top-N   :", TOP_N_COHERENCE)

In [ ]:
R_CANDIDATE_SCRIPT = (
    LDA_DIR
    / "domain_filtered_candidate_evaluation.R"
)

r_candidate_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_dir     <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    30, 50, 70, 100
)

BURN_IN <- 500
ITERATIONS <- 1000
THIN <- 50

TOP_N <- 10

dir.create(
    output_dir,
    recursive = TRUE,
    showWarnings = FALSE
)

# ============================================================
# Load domain-filtered processed corpus
# ============================================================

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ============================================================
# Reconstruct final DTM
# ============================================================

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ============================================================
# Training / held-out split
# ============================================================

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Vocabulary represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove empty documents.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(
    !train_nonempty
)

n_empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ============================================================
# Binary training DTM for semantic coherence
# ============================================================

binary_train <- dtm_train
binary_train$v[] <- 1

term_doc_freq <- slam::col_sums(
    binary_train
)

# ============================================================
# Semantic coherence
# ============================================================

topic_coherence <- function(
    top_indices,
    binary_dtm,
    term_df
) {

    score <- 0

    for (m in 2:length(top_indices)) {

        for (l in 1:(m - 1)) {

            term_m <- top_indices[m]
            term_l <- top_indices[l]

            docs_m <- binary_dtm$i[
                binary_dtm$j == term_m
            ]

            docs_l <- binary_dtm$i[
                binary_dtm$j == term_l
            ]

            cooccur <- length(
                intersect(
                    docs_m,
                    docs_l
                )
            )

            score <- score + log(
                (cooccur + 1)
                / term_df[term_l]
            )
        }
    }

    score
}

# ============================================================
# Model-level results
# ============================================================

model_results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Mean_Coherence = numeric(),
    Median_Coherence = numeric(),
    Min_Coherence = numeric(),
    Mean_Similarity = numeric(),
    Median_Similarity = numeric(),
    Max_Similarity = numeric(),
    Elapsed_seconds = numeric()
)

# ============================================================
# Fit candidate models
# ============================================================

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "...\n"
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    # --------------------------------------------------------
    # Held-out perplexity
    # --------------------------------------------------------

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    # --------------------------------------------------------
    # Topic-word probabilities
    # --------------------------------------------------------

    posterior_model <- topicmodels::posterior(
        lda_model
    )

    beta <- posterior_model$terms

    # --------------------------------------------------------
    # Coherence and top terms
    # --------------------------------------------------------

    coherence_values <- numeric(k)

    top_term_records <- list()

    for (topic_id in seq_len(k)) {

        top_indices <- order(
            beta[
                topic_id,
            ],
            decreasing = TRUE
        )[seq_len(TOP_N)]

        coherence_values[
            topic_id
        ] <- topic_coherence(
            top_indices,
            binary_train,
            term_doc_freq
        )

        top_term_records[[
            topic_id
        ]] <- data.frame(
            Topic = topic_id,
            Rank = seq_len(TOP_N),
            Term = colnames(beta)[
                top_indices
            ],
            Probability = beta[
                topic_id,
                top_indices
            ]
        )
    }

    top_terms <- do.call(
        rbind,
        top_term_records
    )

    # --------------------------------------------------------
    # Topic cosine similarity
    # --------------------------------------------------------

    beta_norm <- beta / sqrt(
        rowSums(
            beta^2
        )
    )

    similarity_matrix <- (
        beta_norm
        %*%
        t(beta_norm)
    )

    similarity_values <- (
        similarity_matrix[
            upper.tri(
                similarity_matrix
            )
        ]
    )

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    # --------------------------------------------------------
    # Model-level diagnostics
    # --------------------------------------------------------

    model_results <- rbind(
        model_results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Mean_Coherence = mean(
                coherence_values
            ),
            Median_Coherence = median(
                coherence_values
            ),
            Min_Coherence = min(
                coherence_values
            ),
            Mean_Similarity = mean(
                similarity_values
            ),
            Median_Similarity = median(
                similarity_values
            ),
            Max_Similarity = max(
                similarity_values
            ),
            Elapsed_seconds = elapsed
        )
    )

    # --------------------------------------------------------
    # Save topic-level coherence
    # --------------------------------------------------------

    write.csv(
        data.frame(
            Topic = seq_len(k),
            Coherence = coherence_values
        ),
        file.path(
            output_dir,
            paste0(
                "coherence_K",
                k,
                ".csv"
            )
        ),
        row.names = FALSE
    )

    # --------------------------------------------------------
    # Save top terms
    # --------------------------------------------------------

    write.csv(
        top_terms,
        file.path(
            output_dir,
            paste0(
                "top_terms_K",
                k,
                ".csv"
            )
        ),
        row.names = FALSE
    )

    # --------------------------------------------------------
    # Save beta
    # --------------------------------------------------------

    beta_df <- data.frame(
        Topic = seq_len(k),
        beta,
        check.names = FALSE
    )

    write.csv(
        beta_df,
        file.path(
            output_dir,
            paste0(
                "beta_K",
                k,
                ".csv"
            )
        ),
        row.names = FALSE
    )

    # Save after every completed candidate.
    write.csv(
        model_results,
        file.path(
            output_dir,
            "candidate_diagnostics.csv"
        ),
        row.names = FALSE
    )

    cat(
        sprintf(
            paste0(
                "K=%d | ",
                "perplexity=%.4f | ",
                "mean coherence=%.4f | ",
                "mean similarity=%.4f | ",
                "%.2f s\n"
            ),
            k,
            heldout_perplexity,
            mean(coherence_values),
            mean(similarity_values),
            elapsed
        )
    )

    flush.console()
}
'''

R_CANDIDATE_SCRIPT.write_text(
    r_candidate_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_CANDIDATE_SCRIPT
)

In [ ]:
corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

candidate_output_dir = (
    LDA_DIR
    / "overton_domain_filtered_candidates"
)

candidate_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Overton candidate evaluation..."
)

print(
    "K values:",
    K_CANDIDATES
)

print(
    "Minimum DF:",
    min_doc_freq
)

candidate_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_CANDIDATE_SCRIPT),
        str(processed_file),
        str(train_file),
        str(candidate_output_dir),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    candidate_run.returncode
)

if candidate_run.stdout.strip():
    print("\nR output:")
    print(candidate_run.stdout)

if candidate_run.stderr.strip():
    print("\nR messages:")
    print(candidate_run.stderr)

In [ ]:
candidate_diagnostics_file = (
    candidate_output_dir
    / "candidate_diagnostics.csv"
)

print(
    "Diagnostics file:",
    candidate_diagnostics_file
)

if candidate_diagnostics_file.exists():

    overton_candidate_diagnostics = pd.read_csv(
        candidate_diagnostics_file
    )

    display(
        overton_candidate_diagnostics
    )

else:

    print(
        "No completed candidate diagnostics were found."
    )

In [ ]:
overton_model_comparison = (
    overton_candidate_diagnostics[
        [
            "K",
            "Perplexity",
            "Mean_Coherence",
            "Mean_Similarity",
        ]
    ]
    .copy()
)

# Lower perplexity is better.
overton_model_comparison[
    "Perplexity_Rank"
] = (
    overton_model_comparison[
        "Perplexity"
    ]
    .rank(
        ascending=True,
        method="min",
    )
)

# Higher / less-negative coherence is better.
overton_model_comparison[
    "Coherence_Rank"
] = (
    overton_model_comparison[
        "Mean_Coherence"
    ]
    .rank(
        ascending=False,
        method="min",
    )
)

# Lower inter-topic similarity is better.
overton_model_comparison[
    "Similarity_Rank"
] = (
    overton_model_comparison[
        "Mean_Similarity"
    ]
    .rank(
        ascending=True,
        method="min",
    )
)

overton_model_comparison[
    "Mean_Rank"
] = (
    overton_model_comparison[
        [
            "Perplexity_Rank",
            "Coherence_Rank",
            "Similarity_Rank",
        ]
    ]
    .mean(axis=1)
)

overton_model_comparison = (
    overton_model_comparison
    .sort_values(
        [
            "Mean_Rank",
            "K",
        ]
    )
    .reset_index(drop=True)
)

overton_model_comparison

#### 5.3.1 Topic Interpretability and Granularity

The numerical diagnostics reveal a trade-off between predictive
performance and semantic coherence. Candidate models are therefore
inspected for thematic interpretability and excessive topic
fragmentation before selecting the final number of topics.

In [ ]:
overton_candidate_top_terms = {}

for k in K_CANDIDATES:

    top_terms_file = (
        candidate_output_dir
        / f"top_terms_K{k}.csv"
    )

    top_terms_df = pd.read_csv(
        top_terms_file
    )

    overton_candidate_top_terms[k] = (
        top_terms_df
    )

    topic_summary = (
        top_terms_df[
            top_terms_df["Rank"] <= 10
        ]
        .groupby("Topic")["Term"]
        .apply(
            lambda terms: ", ".join(terms)
        )
        .reset_index(
            name="Top_Terms"
        )
    )

    print(f"\nK = {k}")
    print("-" * 20)

    display(topic_summary)

### 5.4 Scopus Topic-Number Selection

The same model-selection procedure is applied independently to the
domain-filtered Scopus corpus. A coarse topic-number search is first
performed using short Gibbs chains and held-out perplexity. The search
is subsequently extended if perplexity continues to decrease at the
upper boundary.

In [ ]:
R_SCOPUS_COARSE_SCRIPT = (
    LDA_DIR
    / "domain_filtered_coarse_k_search.R"
)

print(
    "Scopus coarse-search script:",
    R_SCOPUS_COARSE_SCRIPT
)

In [ ]:
corpus_name = "scopus"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

coarse_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_coarse_k_search.csv"
)

n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Scopus coarse K search..."
)

print(
    "Documents       :",
    f"{n_documents:,}"
)

print(
    "Minimum DF      :",
    min_doc_freq
)

print(
    "K values        :",
    K_COARSE
)

scopus_coarse_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_SCOPUS_COARSE_SCRIPT),
        str(processed_file),
        str(train_file),
        str(coarse_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    scopus_coarse_run.returncode
)

if scopus_coarse_run.stdout.strip():

    print("\nR output:")
    print(
        scopus_coarse_run.stdout
    )

if scopus_coarse_run.stderr.strip():

    print("\nR messages:")
    print(
        scopus_coarse_run.stderr
    )

if coarse_output.exists():

    scopus_coarse_results = pd.read_csv(
        coarse_output
    )

    print(
        "\nCompleted K values:",
        scopus_coarse_results[
            "K"
        ].tolist()
    )

    display(
        scopus_coarse_results
    )

else:

    print(
        "\nNo completed Scopus K values were saved."
    )

extended

In [ ]:
R_SCOPUS_EXTENDED_SCRIPT = (
    LDA_DIR
    / "domain_filtered_extended_k_search.R"
)

In [ ]:
K_EXTENDED = [
    60,
    70,
    80,
    100,
]

corpus_name = "scopus"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

extended_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_extended_k_search.csv"
)

n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Scopus extended K search..."
)

print(
    "Documents       :",
    f"{n_documents:,}"
)

print(
    "Minimum DF      :",
    min_doc_freq
)

print(
    "K values        :",
    K_EXTENDED
)

scopus_extended_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_SCOPUS_EXTENDED_SCRIPT),
        str(processed_file),
        str(train_file),
        str(extended_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    scopus_extended_run.returncode
)

if scopus_extended_run.stdout.strip():
    print("\nR output:")
    print(
        scopus_extended_run.stdout
    )

if scopus_extended_run.stderr.strip():
    print("\nR messages:")
    print(
        scopus_extended_run.stderr
    )

if extended_output.exists():

    scopus_extended_results = pd.read_csv(
        extended_output
    )

    print(
        "\nCompleted K values:",
        scopus_extended_results[
            "K"
        ].tolist()
    )

    display(
        scopus_extended_results
    )

else:

    print(
        "\nNo completed extended K values were saved."
    )

#### 5.4.1 Scopus Topic Interpretability and Granularity

The candidate Scopus models are inspected for thematic
interpretability and topic fragmentation. This complements the
quantitative comparison based on held-out perplexity, semantic
coherence, and inter-topic similarity.

In [ ]:
LDA_DIR = OUTPUT_DIR / "lda"

LDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("LDA directory:", LDA_DIR)

In [ ]:
K_CANDIDATES = [
    30,
    50,
    70,
    100,
]

scopus_candidate_output_dir = (
    LDA_DIR
    / "scopus_domain_filtered_candidates"
)

scopus_candidate_diagnostics = pd.read_csv(
    scopus_candidate_output_dir
    / "candidate_diagnostics.csv"
)

display(
    scopus_candidate_diagnostics
)

In [ ]:
scopus_candidate_top_terms = {}

scopus_candidate_output_dir = (
    LDA_DIR
    / "scopus_domain_filtered_candidates"
)

for k in K_CANDIDATES:

    top_terms_file = (
        scopus_candidate_output_dir
        / f"top_terms_K{k}.csv"
    )

    if not top_terms_file.exists():
        raise FileNotFoundError(
            f"Missing file: {top_terms_file}"
        )

    top_terms_df = pd.read_csv(
        top_terms_file
    )

    scopus_candidate_top_terms[
        k
    ] = top_terms_df

    topic_summary = (
        top_terms_df[
            top_terms_df["Rank"] <= 10
        ]
        .groupby(
            "Topic"
        )["Term"]
        .apply(
            lambda terms: ", ".join(terms)
        )
        .reset_index(
            name="Top_Terms"
        )
    )

    print(f"\nK = {k}")
    print("-" * 20)

    display(
        topic_summary
    )

### 5.5 Final Topic-Number Selection

For both corpora, increasing the number of topics improved held-out
perplexity and reduced mean inter-topic similarity, but progressively
reduced semantic coherence and produced increasingly fine-grained topic
fragmentation.

Inspection of candidate topic structures showed that the 30-topic
models combined several substantively distinct themes, whereas the
70- and 100-topic models increasingly divided established themes into
narrower or partially overlapping components. The 50-topic models
provided a suitable balance between predictive performance, semantic
coherence, topic distinctiveness, and substantive interpretability.

Accordingly, the final number of topics was selected independently as
\(K=50\) for both the Overton and Scopus corpora.

## 6. Final LDA Models

Following model selection, the final LDA models are estimated using all
documents in each domain-filtered corpus. The training/held-out split
used for model selection is no longer required at this stage.

Both final models use 50 topics and are estimated using longer Gibbs
chains to obtain more stable topic-word and document-topic
distributions.

In [2]:
import shutil
from pathlib import Path
import math
import pandas as pd
import subprocess

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/nfs/mfirdausi/project/review_paper_2"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "output"
)

R_PREPROCESS_DIR = (
    OUTPUT_DIR
    / "r_preprocessing"
)

LDA_DIR = (
    OUTPUT_DIR
    / "lda"
)

# ---------------------------------------------------------
# R executable
# ---------------------------------------------------------

rscript_path = shutil.which("Rscript")

if rscript_path is None:
    raise FileNotFoundError(
        "Rscript was not found in PATH."
    )

RSCRIPT = Path(
    rscript_path
)

# ---------------------------------------------------------
# Final R script
# ---------------------------------------------------------

R_FINAL_LDA_SCRIPT = (
    LDA_DIR
    / "fit_final_domain_filtered_lda.R"
)

# ---------------------------------------------------------
# Final LDA configuration
# ---------------------------------------------------------

FINAL_K = {
    "overton": 50,
    "scopus": 50,
}

FINAL_BURN_IN = 2000
FINAL_ITERATIONS = 10000
FINAL_THIN = 100
FINAL_RANDOM_SEED = 123

# Final vocabulary rule:
# minimum document frequency = 0.2% of corpus documents.
MIN_DOC_PERCENT = 0.002

# ---------------------------------------------------------
# Validate directories/files
# ---------------------------------------------------------

required_paths = [
    PROJECT_DIR,
    OUTPUT_DIR,
    R_PREPROCESS_DIR,
    LDA_DIR,
    R_FINAL_LDA_SCRIPT,
]

for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {path}"
        )

# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print("Section 6 configuration")
print("-----------------------")

print(
    "Project directory :",
    PROJECT_DIR
)

print(
    "Output directory  :",
    OUTPUT_DIR
)

print(
    "Rscript           :",
    RSCRIPT
)

print(
    "R preprocessing   :",
    R_PREPROCESS_DIR
)

print(
    "LDA directory     :",
    LDA_DIR
)

print(
    "Final R script    :",
    R_FINAL_LDA_SCRIPT
)

print(
    "R script exists   :",
    R_FINAL_LDA_SCRIPT.exists()
)

print()

print(
    "Final K           :",
    FINAL_K
)

print(
    "Burn-in           :",
    FINAL_BURN_IN
)

print(
    "Iterations        :",
    FINAL_ITERATIONS
)

print(
    "Thin              :",
    FINAL_THIN
)

print(
    "Min DF fraction   :",
    MIN_DOC_PERCENT
)

Section 6 configuration
-----------------------
Project directory : /nfs/mfirdausi/project/review_paper_2
Output directory  : /nfs/mfirdausi/project/review_paper_2/output
Rscript           : /nfs/mfirdausi/miniconda3/envs/pytorch/bin/Rscript
R preprocessing   : /nfs/mfirdausi/project/review_paper_2/output/r_preprocessing
LDA directory     : /nfs/mfirdausi/project/review_paper_2/output/lda
Final R script    : /nfs/mfirdausi/project/review_paper_2/output/lda/fit_final_domain_filtered_lda.R
R script exists   : True

Final K           : {'overton': 50, 'scopus': 50}
Burn-in           : 2000
Iterations        : 10000
Thin              : 100
Min DF fraction   : 0.002


### 6.1 Final LDA configuration

In [3]:
FINAL_K = {
    "overton": 50,
    "scopus": 50,
}

FINAL_BURN_IN = 2000
FINAL_ITERATIONS = 10000
FINAL_THIN = 100
FINAL_RANDOM_SEED = 123

print("Final LDA configuration")
print("-----------------------")

for corpus_name, k in FINAL_K.items():
    print(
        f"{corpus_name.upper():8s} : K = {k}"
    )

print()
print("Burn-in    :", FINAL_BURN_IN)
print("Iterations :", FINAL_ITERATIONS)
print("Thin       :", FINAL_THIN)
print("Seed       :", FINAL_RANDOM_SEED)

Final LDA configuration
-----------------------
OVERTON  : K = 50
SCOPUS   : K = 50

Burn-in    : 2000
Iterations : 10000
Thin       : 100
Seed       : 123


### 6.2 Create the final R script

In [4]:
R_FINAL_LDA_SCRIPT = (
    LDA_DIR
    / "fit_final_domain_filtered_lda.R"
)

r_final_lda_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
output_dir     <- args[2]
min_doc_freq   <- as.integer(args[3])
k              <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3

BURN_IN <- 2000
ITERATIONS <- 10000
THIN <- 100
RANDOM_SEED <- 123

TOP_N <- 20

dir.create(
    output_dir,
    recursive = TRUE,
    showWarnings = FALSE
)

# ============================================================
# Load complete processed corpus
# ============================================================

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

original_id <- which(valid)

processed <- processed[
    valid
]

corpus <- VCorpus(
    VectorSource(processed)
)

# ============================================================
# Construct final DTM
# ============================================================

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= min_doc_freq
]

# Remove any documents that become empty after vocabulary filtering.
nonempty <- (
    slam::row_sums(dtm) > 0
)

original_id <- original_id[
    nonempty
]

dtm <- dtm[
    nonempty,
]

cat(
    "Documents:",
    dtm$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm$ncol,
    "\n"
)

cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)

cat(
    "K:",
    k,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n\n"
)

# ============================================================
# Final Gibbs LDA
# ============================================================

start_time <- Sys.time()

lda_model <- topicmodels::LDA(
    dtm,
    k = k,
    method = "Gibbs",
    control = list(
        seed = RANDOM_SEED,
        burnin = BURN_IN,
        iter = ITERATIONS,
        thin = THIN
    )
)

elapsed <- as.numeric(
    difftime(
        Sys.time(),
        start_time,
        units = "secs"
    )
)

posterior_model <- topicmodels::posterior(
    lda_model
)

beta <- posterior_model$terms
theta <- posterior_model$topics

# ============================================================
# Save beta
# ============================================================

beta_df <- data.frame(
    Topic = seq_len(k),
    beta,
    check.names = FALSE
)

write.csv(
    beta_df,
    file.path(
        output_dir,
        "beta.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Save theta
# ============================================================

theta_df <- data.frame(
    document_id = original_id,
    theta,
    check.names = FALSE
)

write.csv(
    theta_df,
    file.path(
        output_dir,
        "theta.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Top terms
# ============================================================

top_term_records <- list()

for (topic_id in seq_len(k)) {

    ord <- order(
        beta[
            topic_id,
        ],
        decreasing = TRUE
    )[seq_len(TOP_N)]

    top_term_records[[
        topic_id
    ]] <- data.frame(
        Topic = topic_id,
        Rank = seq_len(TOP_N),
        Term = colnames(beta)[
            ord
        ],
        Probability = beta[
            topic_id,
            ord
        ]
    )
}

top_terms <- do.call(
    rbind,
    top_term_records
)

write.csv(
    top_terms,
    file.path(
        output_dir,
        "top_terms.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Dominant topic
# ============================================================

dominant_topic <- max.col(
    theta,
    ties.method = "first"
)

dominant_probability <- apply(
    theta,
    1,
    max
)

document_topics <- data.frame(
    document_id = original_id,
    Dominant_Topic = dominant_topic,
    Dominant_Probability = dominant_probability
)

write.csv(
    document_topics,
    file.path(
        output_dir,
        "document_topics.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Topic prevalence
# ============================================================

topic_prevalence <- colMeans(
    theta
)

prevalence_df <- data.frame(
    Topic = seq_len(k),
    Mean_Probability = topic_prevalence
)

prevalence_df <- prevalence_df[
    order(
        prevalence_df$Mean_Probability,
        decreasing = TRUE
    ),
]

write.csv(
    prevalence_df,
    file.path(
        output_dir,
        "topic_prevalence.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Final summary
# ============================================================

summary_df <- data.frame(
    Documents = dtm$nrow,
    Vocabulary = dtm$ncol,
    Tokens = sum(dtm$v),
    K = k,
    Minimum_DF = min_doc_freq,
    Burn_in = BURN_IN,
    Iterations = ITERATIONS,
    Thin = THIN,
    Seed = RANDOM_SEED,
    Elapsed_seconds = elapsed
)

write.csv(
    summary_df,
    file.path(
        output_dir,
        "model_summary.csv"
    ),
    row.names = FALSE
)

cat(
    "Final LDA completed in",
    round(elapsed, 2),
    "seconds.\n"
)
'''

R_FINAL_LDA_SCRIPT.write_text(
    r_final_lda_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_FINAL_LDA_SCRIPT
)

Created: /nfs/mfirdausi/project/review_paper_2/output/lda/fit_final_domain_filtered_lda.R


### 6.3 Final Overton LDA Model

The final Overton LDA model is estimated on the complete
domain-filtered corpus using \(K=50\). The longer Gibbs chain is used
for final estimation after completion of topic-number selection.

In [5]:
corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

dtm_summary_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_final_dtm_summary.csv"
)

final_output_dir = (
    LDA_DIR
    / "final_overton_k50"
)

final_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------
# Load final DTM information from disk
# ---------------------------------------------------------

dtm_summary = pd.read_csv(
    dtm_summary_file
)

n_documents = int(
    dtm_summary.loc[
        0,
        "Documents"
    ]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

k = FINAL_K[
    corpus_name
]

# ---------------------------------------------------------
# Validate required files
# ---------------------------------------------------------

for required_file in [
    processed_file,
    dtm_summary_file,
    R_FINAL_LDA_SCRIPT,
]:

    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing required file: {required_file}"
        )

# ---------------------------------------------------------
# Final Overton LDA
# ---------------------------------------------------------

print("Running final Overton LDA...")
print("-----------------------------")
print("Documents  :", f"{n_documents:,}")
print("K          :", k)
print("Minimum DF :", min_doc_freq)
print("Burn-in    :", FINAL_BURN_IN)
print("Iterations :", FINAL_ITERATIONS)
print("Thin       :", FINAL_THIN)

final_overton_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_FINAL_LDA_SCRIPT),
        str(processed_file),
        str(final_output_dir),
        str(min_doc_freq),
        str(k),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "\nReturn code:",
    final_overton_run.returncode
)

if final_overton_run.stdout.strip():

    print("\nR output:")
    print(
        final_overton_run.stdout
    )

if final_overton_run.stderr.strip():

    print("\nR messages:")
    print(
        final_overton_run.stderr
    )

Running final Overton LDA...
-----------------------------
Documents  : 5,045
K          : 50
Minimum DF : 11
Burn-in    : 2000
Iterations : 10000
Thin       : 100

Return code: 0

R output:
Documents: 5045 
Vocabulary: 2865 
Tokens: 512540 
K: 50 
Minimum DF: 11 

Final LDA completed in 397.06 seconds.



### 6.4 Final Scopus LDA Model

The final Scopus LDA model is estimated on the complete
domain-filtered corpus using \(K=50\) and the same final Gibbs-sampling
configuration used for Overton.

In [8]:
corpus_name = "scopus"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

dtm_summary_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_final_dtm_summary.csv"
)

final_output_dir = (
    LDA_DIR
    / "final_scopus_k50"
)

final_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------
# Load final DTM information
# ---------------------------------------------------------

dtm_summary = pd.read_csv(
    dtm_summary_file
)

n_documents = int(
    dtm_summary.loc[
        0,
        "Documents"
    ]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

k = FINAL_K[
    corpus_name
]

# ---------------------------------------------------------
# Validate required files
# ---------------------------------------------------------

for required_file in [
    processed_file,
    dtm_summary_file,
    R_FINAL_LDA_SCRIPT,
]:

    if not required_file.exists():

        raise FileNotFoundError(
            f"Missing required file: {required_file}"
        )

# ---------------------------------------------------------
# Final Scopus LDA
# ---------------------------------------------------------

print("Running final Scopus LDA...")
print("----------------------------")
print("Documents  :", f"{n_documents:,}")
print("K          :", k)
print("Minimum DF :", min_doc_freq)
print("Burn-in    :", FINAL_BURN_IN)
print("Iterations :", FINAL_ITERATIONS)
print("Thin       :", FINAL_THIN)

final_scopus_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_FINAL_LDA_SCRIPT),
        str(processed_file),
        str(final_output_dir),
        str(min_doc_freq),
        str(k),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "\nReturn code:",
    final_scopus_run.returncode
)

if final_scopus_run.stdout.strip():

    print("\nR output:")
    print(
        final_scopus_run.stdout
    )

if final_scopus_run.stderr.strip():

    print("\nR messages:")
    print(
        final_scopus_run.stderr
    )

Running final Scopus LDA...
----------------------------
Documents  : 12,042
K          : 50
Minimum DF : 25
Burn-in    : 2000
Iterations : 10000
Thin       : 100

Return code: 0

R output:
Documents: 12042 
Vocabulary: 2737 
Tokens: 1384122 
K: 50 
Minimum DF: 25 

Final LDA completed in 1096.87 seconds.



## 7. Topic Analysis and Cross-Corpus Comparison

### 7.1 Topic Interpretation and Labeling


#### 7.1.1 Load Final LDA Outputs

In [9]:
# ---------------------------------------------------------
# Final model directories
# ---------------------------------------------------------

FINAL_MODEL_DIRS = {
    "overton": (
        LDA_DIR
        / "final_overton_k50"
    ),
    "scopus": (
        LDA_DIR
        / "final_scopus_k50"
    ),
}

final_lda = {}

for corpus_name, model_dir in FINAL_MODEL_DIRS.items():

    required_files = {
        "summary": model_dir / "model_summary.csv",
        "beta": model_dir / "beta.csv",
        "theta": model_dir / "theta.csv",
        "top_terms": model_dir / "top_terms.csv",
        "document_topics": model_dir / "document_topics.csv",
        "prevalence": model_dir / "topic_prevalence.csv",
    }

    # Check that all outputs exist.
    for name, file_path in required_files.items():

        if not file_path.exists():

            raise FileNotFoundError(
                f"Missing {corpus_name} {name}: "
                f"{file_path}"
            )

    final_lda[corpus_name] = {
        name: pd.read_csv(file_path)
        for name, file_path
        in required_files.items()
    }

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        "Beta shape          :",
        final_lda[corpus_name]["beta"].shape
    )

    print(
        "Theta shape         :",
        final_lda[corpus_name]["theta"].shape
    )

    print(
        "Top-term rows       :",
        len(
            final_lda[
                corpus_name
            ]["top_terms"]
        )
    )

    print(
        "Document-topic rows :",
        len(
            final_lda[
                corpus_name
            ]["document_topics"]
        )
    )

    print(
        "Prevalence rows     :",
        len(
            final_lda[
                corpus_name
            ]["prevalence"]
        )
    )

    print()

OVERTON
-------
Beta shape          : (50, 2866)
Theta shape         : (5045, 51)
Top-term rows       : 1000
Document-topic rows : 5045
Prevalence rows     : 50

SCOPUS
------
Beta shape          : (50, 2738)
Theta shape         : (12042, 51)
Top-term rows       : 1000
Document-topic rows : 12042
Prevalence rows     : 50



#### 7.1.2 Validate Topic Distributions

In [10]:
for corpus_name, outputs in final_lda.items():

    beta = (
        outputs["beta"]
        .drop(
            columns=["Topic"]
        )
    )

    theta = (
        outputs["theta"]
        .drop(
            columns=["document_id"]
        )
    )

    beta_row_sums = (
        beta.sum(axis=1)
    )

    theta_row_sums = (
        theta.sum(axis=1)
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        "Beta row sums  :",
        f"{beta_row_sums.min():.6f}",
        "to",
        f"{beta_row_sums.max():.6f}",
    )

    print(
        "Theta row sums :",
        f"{theta_row_sums.min():.6f}",
        "to",
        f"{theta_row_sums.max():.6f}",
    )

    print()

OVERTON
-------
Beta row sums  : 1.000000 to 1.000000
Theta row sums : 1.000000 to 1.000000

SCOPUS
------
Beta row sums  : 1.000000 to 1.000000
Theta row sums : 1.000000 to 1.000000



#### 7.1.3 Top Terms

In [11]:
TOP_TERMS_DISPLAY = 10

final_topic_terms = {}

for corpus_name, outputs in final_lda.items():

    top_terms = (
        outputs["top_terms"]
        .copy()
    )

    topic_terms = (
        top_terms[
            top_terms["Rank"]
            <= TOP_TERMS_DISPLAY
        ]
        .sort_values(
            [
                "Topic",
                "Rank",
            ]
        )
        .groupby(
            "Topic"
        )["Term"]
        .apply(
            lambda x: ", ".join(x)
        )
        .reset_index(
            name="Top_Terms"
        )
    )

    final_topic_terms[
        corpus_name
    ] = topic_terms

    print()
    print(corpus_name.upper())
    print("=" * len(corpus_name))

    display(
        topic_terms
    )


OVERTON


,Topic,Top_Terms
0,1,"oper, system, flexibl, plan, integr, consid, c..."
1,2,"test, condit, result, use, wave, perform, expe..."
2,3,"effect, factor, rate, increas, depend, also, m..."
3,4,"design, effici, perform, can, improv, hybrid, ..."
4,5,"applic, develop, paper, standard, includ, new,..."
5,6,"term, main, author, relat, concept, take, one,..."
6,7,"approach, framework, can, present, process, st..."
7,8,"polici, electr, countri, sector, industri, sup..."
8,9,"develop, project, engin, research, nation, rig..."
9,10,"generat, reserv, unit, variabl, penetr, dispat..."



SCOPUS


,Topic,Top_Terms
0,1,"network, neural, train, use, predict, deep, ar..."
1,2,"voltag, loss, reactiv, activ, bus, ieee, regul..."
2,3,"estim, state, measur, paramet, method, use, ac..."
3,4,"market, energi, price, electr, trade, mechan, ..."
4,5,"can, will, due, issu, one, also, howev, may, m..."
5,6,"model, develop, propos, simul, linear, datadri..."
6,7,"microgrid, oper, manag, system, generat, mode,..."
7,8,"system, reserv, ltd, engin, right, publish, te..."
8,9,"technolog, energi, intellig, sustain, artifici..."
9,10,"techniqu, use, differ, perform, various, sugge..."


#### 7.1.4 Representative Documents

In [31]:
for corpus_name in [
    "overton",
    "scopus",
]:

    saved_abstracts_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_abstracts.csv"
    )

    # Exact abstracts originally supplied to R.
    saved_abstracts = pd.read_csv(
        saved_abstracts_file,
        keep_default_na=False,
    )

    # Metadata corpus reconstructed using the frozen
    # domain-relevance screening.
    reconstructed = (
        lda_corpora_filtered[
            corpus_name
        ]
        .reset_index(drop=True)
        .copy()
    )

    saved_text = (
        saved_abstracts["Abstract"]
        .fillna("")
        .astype(str)
        .str.strip()
        .reset_index(drop=True)
    )

    reconstructed_text = (
        reconstructed["Abstract"]
        .fillna("")
        .astype(str)
        .str.strip()
        .reset_index(drop=True)
    )

    same_length = (
        len(saved_text)
        == len(reconstructed_text)
    )

    if same_length:

        comparison = (
            saved_text
            == reconstructed_text
        )

        n_matching = int(
            comparison.sum()
        )

        n_different = int(
            (~comparison).sum()
        )

        exact_order_match = (
            n_different == 0
        )

    else:

        n_matching = 0
        n_different = abs(
            len(saved_text)
            - len(reconstructed_text)
        )

        exact_order_match = False

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Saved LDA abstracts : "
        f"{len(saved_text):,}"
    )

    print(
        f"Reconstructed       : "
        f"{len(reconstructed_text):,}"
    )

    print(
        f"Matching rows       : "
        f"{n_matching:,}"
    )

    print(
        f"Different rows      : "
        f"{n_different:,}"
    )

    print(
        "Exact order match   :",
        exact_order_match
    )

    print()

OVERTON
-------
Saved LDA abstracts : 5,045
Reconstructed       : 5,045
Matching rows       : 5,045
Different rows      : 0
Exact order match   : True

SCOPUS
------
Saved LDA abstracts : 12,042
Reconstructed       : 12,042
Matching rows       : 12,042
Different rows      : 0
Exact order match   : True



In [32]:
FILTERED_METADATA_DIR = (
    OUTPUT_DIR
    / "domain_filtered_metadata"
)

FILTERED_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

final_metadata = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    metadata = (
        lda_corpora_filtered[
            corpus_name
        ]
        .reset_index(drop=True)
        .copy()
    )

    # R uses 1-based document IDs.
    metadata[
        "document_id"
    ] = (
        metadata.index
        + 1
    )

    final_metadata[
        corpus_name
    ] = metadata

    output_file = (
        FILTERED_METADATA_DIR
        / f"{corpus_name}_domain_filtered_metadata.csv"
    )

    metadata.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents : "
        f"{len(metadata):,}"
    )

    print(
        "Saved     :",
        output_file
    )

    print()

OVERTON
-------
Documents : 5,045
Saved     : /nfs/mfirdausi/project/review_paper_2/output/domain_filtered_metadata/overton_domain_filtered_metadata.csv

SCOPUS
------
Documents : 12,042
Saved     : /nfs/mfirdausi/project/review_paper_2/output/domain_filtered_metadata/scopus_domain_filtered_metadata.csv



In [33]:
TOP_REPRESENTATIVE_DOCS = 5

representative_documents = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    theta = (
        final_lda[
            corpus_name
        ]["theta"]
        .copy()
    )

    metadata = (
        final_metadata[
            corpus_name
        ]
        .copy()
    )

    # Topic-probability columns in theta.csv.
    topic_columns = [
        column
        for column in theta.columns
        if column != "document_id"
    ]

    records = []

    for topic_number, topic_column in enumerate(
        topic_columns,
        start=1,
    ):

        topic_scores = (
            theta[
                [
                    "document_id",
                    topic_column,
                ]
            ]
            .rename(
                columns={
                    topic_column:
                    "Topic_Probability"
                }
            )
        )

        topic_scores = (
            topic_scores
            .merge(
                metadata,
                on="document_id",
                how="left",
                validate="one_to_one",
            )
        )

        top_documents = (
            topic_scores
            .nlargest(
                TOP_REPRESENTATIVE_DOCS,
                "Topic_Probability",
            )
            .copy()
        )

        top_documents[
            "Topic"
        ] = topic_number

        top_documents[
            "Representative_Rank"
        ] = range(
            1,
            len(top_documents) + 1
        )

        records.append(
            top_documents
        )

    representative_df = (
        pd.concat(
            records,
            ignore_index=True,
        )
    )

    representative_documents[
        corpus_name
    ] = representative_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        "Topics                  :",
        len(topic_columns)
    )

    print(
        "Documents/topic         :",
        TOP_REPRESENTATIVE_DOCS
    )

    print(
        "Representative doc rows :",
        len(representative_df)
    )

    print()

OVERTON
-------
Topics                  : 50
Documents/topic         : 5
Representative doc rows : 250

SCOPUS
------
Topics                  : 50
Documents/topic         : 5
Representative doc rows : 250



In [34]:
for corpus_name in [
    "overton",
    "scopus",
]:

    print()
    print(corpus_name.upper())
    print("=" * len(corpus_name))

    topic_1 = (
        representative_documents[
            corpus_name
        ]
        .query(
            "Topic == 1"
        )
    )

    display_columns = [
        column
        for column in [
            "Topic",
            "Representative_Rank",
            "Topic_Probability",
            "Title",
            "Year",
            "DOI",
        ]
        if column in topic_1.columns
    ]

    display(
        topic_1[
            display_columns
        ]
    )


OVERTON


,Topic,Representative_Rank,Topic_Probability,Title,Year,DOI
0,1,1,0.242424,Understanding barriers to utilising flexibilit...,2023,10.1016/j.enpol.2023.113618
1,1,2,0.181818,Flexible distributed multienergy generation sy...,2016,10.1109/TSG.2015.2411392
2,1,3,0.167539,Alternatives No More: Wind and Solar Power Are...,2015,10.1109/MPE.2015.2462311
3,1,4,0.161290,Economic assessment of integrating fast-chargi...,2023,10.1016/j.segan.2023.101083
4,1,5,0.157576,Capacity Market Model Considering Flexible Res...,2018,10.1109/PESGM.2018.8586189



SCOPUS


,Topic,Representative_Rank,Topic_Probability,Title,Year,DOI
0,1,1,0.234043,Physics-informed machine learning with optimiz...,2024,10.1016/j.ijepes.2023.109741
1,1,2,0.209677,Enriching Neural Network Training Dataset to I...,2023,10.1109/PowerTech55446.2023.10202770
2,1,3,0.208861,FAULT DIAGNOSIS MODELLING OF POWER SYSTEM CONT...,2025,10.2316/J.2025.201-0437
3,1,4,0.207101,Physics-Informed Neural Networks for AC Optima...,2022,10.1016/j.epsr.2022.108412
4,1,5,0.206897,DeepOPF-AL: Augmented Learning for Solving AC-...,2023,10.1145/3575813.3576874


In [35]:
INTERPRETATION_DIR = (
    OUTPUT_DIR
    / "topic_interpretation"
)

INTERPRETATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for corpus_name, representative_df in (
    representative_documents.items()
):

    output_file = (
        INTERPRETATION_DIR
        / f"{corpus_name}_representative_documents.csv"
    )

    representative_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():8s}: "
        f"{output_file}"
    )

OVERTON : /nfs/mfirdausi/project/review_paper_2/output/topic_interpretation/overton_representative_documents.csv
SCOPUS  : /nfs/mfirdausi/project/review_paper_2/output/topic_interpretation/scopus_representative_documents.csv


#### 7.1.5 Topic Labeling

In [36]:
topic_interpretation_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    # ---------------------------------------------------------
    # Top terms
    # ---------------------------------------------------------

    terms = (
        final_topic_terms[
            corpus_name
        ]
        .copy()
    )

    # ---------------------------------------------------------
    # Representative documents
    # ---------------------------------------------------------

    reps = (
        representative_documents[
            corpus_name
        ]
        .copy()
    )

    representative_titles = (
        reps
        .sort_values(
            [
                "Topic",
                "Representative_Rank",
            ]
        )
        .groupby(
            "Topic"
        )["Title"]
        .apply(
            lambda titles:
            " | ".join(
                titles
                .fillna("")
                .astype(str)
            )
        )
        .reset_index(
            name="Representative_Titles"
        )
    )

    # ---------------------------------------------------------
    # Combine interpretation evidence
    # ---------------------------------------------------------

    interpretation = (
        terms
        .merge(
            representative_titles,
            on="Topic",
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "Topic"
        )
        .reset_index(drop=True)
    )

    # Labels will be assigned after interpretation.
    interpretation[
        "Topic_Label"
    ] = ""

    topic_interpretation_tables[
        corpus_name
    ] = interpretation

    print()
    print(corpus_name.upper())
    print("=" * len(corpus_name))

    display(
        interpretation
    )


OVERTON


,Topic,Top_Terms,Representative_Titles,Topic_Label
0,1,"oper, system, flexibl, plan, integr, consid, c...",Understanding barriers to utilising flexibilit...,
1,2,"test, condit, result, use, wave, perform, expe...",PREDICTION OF HEAVE AND PITCH LOW FREQUENCY WA...,
2,3,"effect, factor, rate, increas, depend, also, m...",Digitalization as a trigger for a rebound effe...,
3,4,"design, effici, perform, can, improv, hybrid, ...",Design considerations of a transverse flux mac...,
4,5,"applic, develop, paper, standard, includ, new,...",OpenDSS and OpenDSS-PM open source libraries f...,
5,6,"term, main, author, relat, concept, take, one,...",Techniques to Locate the Origin of Power Quali...,
6,7,"approach, framework, can, present, process, st...",SNARKs for C: Verifying program executions suc...,
7,8,"polici, electr, countri, sector, industri, sup...",Political autonomy and resistance in electrici...,
8,9,"develop, project, engin, research, nation, rig...",Managing the NIH Bethesda Campus Capital Asset...,
9,10,"generat, reserv, unit, variabl, penetr, dispat...",An a priori analytical method for the determin...,



SCOPUS


,Topic,Top_Terms,Representative_Titles,Topic_Label
0,1,"network, neural, train, use, predict, deep, ar...",Physics-informed machine learning with optimiz...,
1,2,"voltag, loss, reactiv, activ, bus, ieee, regul...",Resource management with kernel-based approach...,
2,3,"estim, state, measur, paramet, method, use, ac...",Robust Matrix Completion State Estimation in D...,
3,4,"market, energi, price, electr, trade, mechan, ...",Peer-to-Peer Energy Trading Enabled Optimal De...,
4,5,"can, will, due, issu, one, also, howev, may, m...",A survey on power system blackout and cascadin...,
5,6,"model, develop, propos, simul, linear, datadri...",Data-Driven-Aided Linear Three-Phase Power Flo...,
6,7,"microgrid, oper, manag, system, generat, mode,...",Networked Microgrids Framework | Optimal gener...,
7,8,"system, reserv, ltd, engin, right, publish, te...",A distributed data-driven modelling framework ...,
8,9,"technolog, energi, intellig, sustain, artifici...",Digital technologies for a net-zero energy fut...,
9,10,"techniqu, use, differ, perform, various, sugge...",Optimized FACTS Devices for Power System Enhan...,


In [37]:
def show_topic(
    corpus_name,
    topic_number,
):

    terms = (
        topic_interpretation_tables[
            corpus_name
        ]
        .query(
            "Topic == @topic_number"
        )
        .iloc[0]
    )

    reps = (
        representative_documents[
            corpus_name
        ]
        .query(
            "Topic == @topic_number"
        )
        .sort_values(
            "Representative_Rank"
        )
    )

    print(
        f"{corpus_name.upper()} — "
        f"Topic {topic_number}"
    )

    print("=" * 70)

    print("\nTop terms:")
    print(
        terms["Top_Terms"]
    )

    print(
        "\nRepresentative publications:"
    )

    for _, row in reps.iterrows():

        print(
            f"\n{int(row['Representative_Rank'])}. "
            f"{row['Title']}"
        )

        print(
            f"   Topic probability: "
            f"{row['Topic_Probability']:.4f}"
        )

In [38]:
show_topic(
    "overton",
    1,
)

OVERTON — Topic 1

Top terms:
oper, system, flexibl, plan, integr, consid, case, decis, level, requir

Representative publications:

1. Understanding barriers to utilising flexibility in operation and planning of the electricity distribution system – Classification frameworks with applications to Norway
   Topic probability: 0.2424

2. Flexible distributed multienergy generation system expansion planning under uncertainty
   Topic probability: 0.1818

3. Alternatives No More: Wind and Solar Power Are Mainstays of a Clean, Reliable, Affordable Grid
   Topic probability: 0.1675

4. Economic assessment of integrating fast-charging stations and energy communities in grid planning
   Topic probability: 0.1613

5. Capacity Market Model Considering Flexible Resource Requirements
   Topic probability: 0.1576


In [39]:
show_topic(
    "scopus",
    1,
)

SCOPUS — Topic 1

Top terms:
network, neural, train, use, predict, deep, artifici, accuraci, ann, approach

Representative publications:

1. Physics-informed machine learning with optimization-based guarantees: Applications to AC power flow
   Topic probability: 0.2340

2. Enriching Neural Network Training Dataset to Improve Worst-Case Performance Guarantees
   Topic probability: 0.2097

3. FAULT DIAGNOSIS MODELLING OF POWER SYSTEM CONTROLLER BASED ON PLC TECHNOLOGY
   Topic probability: 0.2089

4. Physics-Informed Neural Networks for AC Optimal Power Flow
   Topic probability: 0.2071

5. DeepOPF-AL: Augmented Learning for Solving AC-OPF Problems with a Multi-Valued Load-Solution Mapping
   Topic probability: 0.2069


In [40]:
INTERPRETATION_DIR = (
    OUTPUT_DIR
    / "topic_interpretation"
)

for corpus_name, table in (
    topic_interpretation_tables.items()
):

    output_file = (
        INTERPRETATION_DIR
        / f"{corpus_name}_topic_interpretation.csv"
    )

    table.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():8s}: "
        f"{output_file}"
    )

OVERTON : /nfs/mfirdausi/project/review_paper_2/output/topic_interpretation/overton_topic_interpretation.csv
SCOPUS  : /nfs/mfirdausi/project/review_paper_2/output/topic_interpretation/scopus_topic_interpretation.csv


In [42]:
def show_topic_range(
    corpus_name,
    start_topic,
    end_topic,
):

    for topic_number in range(
        start_topic,
        end_topic + 1,
    ):

        show_topic(
            corpus_name,
            topic_number,
        )

        print("\n")

In [43]:
show_topic_range(
    "overton",
    2,
    10,
)

OVERTON — Topic 2

Top terms:
test, condit, result, use, wave, perform, experi, present, evalu, simul

Representative publications:

1. PREDICTION OF HEAVE AND PITCH LOW FREQUENCY WAVE FORCES AND MOTIONS OF A SEMI-SUBMERSIBLE FLOATING WIND TURBINE AND COMPARISON WITH MODEL TEST DATA
   Topic probability: 0.3665

2. MARINET experiment KNSWING testing an I-Beam OWC attenuator
   Topic probability: 0.3385

3. Real-time hybrid model testing of a braceless semi-submersible wind turbine. Part II: Experimental results
   Topic probability: 0.3082

4. Optimization-based calibration of hydrodynamic drag coefficients for a semisubmersible platform using experimental data of an irregular sea state
   Topic probability: 0.3041

5. Uncertainty assessment of CFD investigation of the nonlinear difference-frequency wave loads on a semisubmersible FOWT platform
   Topic probability: 0.2905


OVERTON — Topic 3

Top terms:
effect, factor, rate, increas, depend, also, may, signific, influenc, result

Repr

In [44]:
show_topic_range(
    "scopus",
    2,
    10,
)

SCOPUS — Topic 2

Top terms:
voltag, loss, reactiv, activ, bus, ieee, regul, deviat, load, profil

Representative publications:

1. Resource management with kernel-based approaches for grid-connected solar photovoltaic systems
   Topic probability: 0.2583

2. Minimization of L-index using genetic algorithm for improvement of voltage profile in power systems
   Topic probability: 0.2460

3. A Zonal Volt/VAR Control Mechanism for High PV Penetration Distribution Systems
   Topic probability: 0.2273

4. A multi-objective volt-var control strategy for distribution networks with high PV penetration
   Topic probability: 0.2229

5. Long-Term Voltage Stability Bifurcation Analysis and Control Considering OLTC Adjustment and Photovoltaic Power Station
   Topic probability: 0.2199


SCOPUS — Topic 3

Top terms:
estim, state, measur, paramet, method, use, accur, identif, accuraci, condit

Representative publications:

1. Robust Matrix Completion State Estimation in Distribution Systems
   Topic 

In [46]:
show_topic_range(
    "overton",
    11,
    20,
)

OVERTON — Topic 11

Top terms:
grid, smart, communic, secur, network, attack, infrastructur, ieee, inform, manag

Representative publications:

1. Smart grid - The new and improved power grid: A survey
   Topic probability: 0.2941

2. An analysis of smart grid attacks and countermeasures
   Topic probability: 0.2689

3. Current and Future Communication Solutions for Smart Grids: A Review
   Topic probability: 0.2577

4. Heterogeneous communication architecture for the smart grid
   Topic probability: 0.2516

5. A survey on smart power grid: frameworks, tools, security issues, and solutions
   Topic probability: 0.2361


OVERTON — Topic 12

Top terms:
electr, vehicl, charg, transport, drive, station, pev, batteri, infrastructur, plugin

Representative publications:

1. Energy efficiency trade-offs in small to large electric vehicles
   Topic probability: 0.3135

2. Traffic impacts on energy consumption of electric and conventional vehicles
   Topic probability: 0.3066

3. Quantifying th

In [47]:
show_topic_range(
    "scopus",
    11,
    20,
)

SCOPUS — Topic 11

Top terms:
oper, flexibl, coordin, system, level, provid, framework, resourc, can, increas

Representative publications:

1. Tracing, Ranking and Valuation of Aggregated der Flexibility in Active Distribution Networks
   Topic probability: 0.2340

2. A hierarchical scheduling framework for DSO and shiftable load aggregator
   Topic probability: 0.2126

3. TensorConvolutionPlus: A python package for distribution system flexibility area estimation
   Topic probability: 0.2059

4. Tso-dso coordination schemes to facilitate distributed resources integration
   Topic probability: 0.1946

5. Comprehensive review of transmission system operators–Distribution system operators collaboration for flexible grid operations
   Topic probability: 0.1919


SCOPUS — Topic 12

Top terms:
framework, across, limit, practic, deploy, integr, remain, scalabl, evalu, valid

Representative publications:

1. STORM-OPF: Robustness Benchmarking for AC-OPF Methods
   Topic probability: 0.2409

2

In [48]:
overton_topic_labels = {
    1: "Power-System Flexibility and Grid Planning",
    2: "Offshore Wind and Marine Energy Modeling",
    3: "Energy and Environmental Impact Factors",
    4: "Electric Machine and Wind Generator Design",
    5: "Power-System Modeling Standards and Software Tools",
    6: "Energy-System Concepts and Technology Assessment",
    7: "Computational Frameworks and Algorithmic Methods",
    8: "Electricity-Sector Policy and Market Reform",
    9: "Energy Research Infrastructure and Nuclear/Fusion Technology",
    10: "Operating Reserves and Renewable Balancing",
}

scopus_topic_labels = {
    1: "Neural Networks and Physics-Informed Learning",
    2: "Voltage and Reactive Power Control",
    3: "Power-System State Estimation",
    4: "Electricity Markets and Peer-to-Peer Energy Trading",
    5: "Grid Reliability, Risk, and Stability Challenges",
    6: "Data-Driven and Linear Power-Flow Modeling",
    7: "Microgrid Operation and Energy Management",
    8: "Power-System Engineering Applications",
    9: "Digitalization and Intelligent Energy Systems",
    10: "Metaheuristic Optimization Methods",
}

overton_topic_labels.update({
    11: "Smart-Grid Communications and Cybersecurity",
    12: "Electric Vehicles and Charging Infrastructure",
    13: "Ancillary Services and Demand-Side Flexibility",
    14: "Comparative Energy-System Performance Analysis",
    15: "Climate Change and Water-Energy Nexus",
    16: "Renewable and Distributed Grid Integration Challenges",
    17: "Power-System Modeling and Simulation",
    18: "Life-Cycle and Environmental Impact Assessment",
    19: "Distributed-Energy Adoption and Energy Equity",
    20: "Distributed Microgrid Control and Multi-Agent Systems",
})

scopus_topic_labels.update({
    11: "Grid Flexibility and TSO-DSO Coordination",
    12: "Power-System Optimization Frameworks and Validation",
    13: "FACTS Device Placement and Optimization",
    14: "Distribution-System Planning and Evaluation Methods",
    15: "Renewable Generation and Siting Assessment",
    16: "Energy and Power Forecasting",
    17: "Smart-Grid Technologies and Applications",
    18: "Multi-Objective and Evolutionary Optimization",
    19: "Energy Efficiency and Decarbonization",
    20: "Cost and Loss Optimization in Power Systems",
})

In [49]:
show_topic_range(
    "overton",
    21,
    30,
)

OVERTON — Topic 21

Top terms:
system, paper, reliabl, ieee, provid, present, integr, interconnect, becom, overal

Representative publications:

1. Reliability and availability modelling of combined heat and power (CHP) systems
   Topic probability: 0.1515

2. Framework of a benchmark testbed for power system cyber-physical reliability studies
   Topic probability: 0.1379

3. Reliability of power system considering replacement of conventional power plants with renewables
   Topic probability: 0.1295

4. Excitation system models for power system stability studies: IEEE committee report
   Topic probability: 0.1203

5. Coarse-grained distributed optimal power flow
   Topic probability: 0.1190


OVERTON — Topic 22

Top terms:
problem, optim, solut, propos, algorithm, comput, solv, formul, ieee, constraint

Representative publications:

1. A Laplacian-Based Approach for Finding Near Globally Optimal Solutions to OPF Problems
   Topic probability: 0.3933

2. Semidefinite programming for opt

In [50]:
show_topic_range(
    "scopus",
    21,
    30,
)

SCOPUS — Topic 21

Top terms:
distribut, direct, algorithm, altern, central, local, converg, decentr, method, propos

Representative publications:

1. Impact of communication delay on asynchronous distributed optimal power flow using ADMM
   Topic probability: 0.3476

2. A Fully-Decentralized Consensus-Based ADMM Approach for DC-OPF with Demand Response
   Topic probability: 0.3350

3. A fully distributed asynchronous approach for multi-area coordinated network-constrained unit commitment
   Topic probability: 0.3178

4. Distributed Energy Optimization in MAS-based Microgrids using Asynchronous ADMM
   Topic probability: 0.2835

5. Consensus ADMM and Proximal ADMM for economic dispatch and AC OPF with SOCP relaxation
   Topic probability: 0.2832


SCOPUS — Topic 22

Top terms:
load, demand, electr, respons, peak, consumpt, suppli, util, manag, use

Representative publications:

1. Intelligent Demand Side Management for Exhaustive Techno-Economic Analysis of Microgrid System
   Topic pr

In [51]:
overton_topic_labels.update({
    21: "Power-System Reliability and Stability",
    22: "Convex Relaxations and Mathematical Programming for OPF",
    23: "Wind and Solar Resource Variability",
    24: "Energy Storage Technologies and Grid Integration",
    25: "Power-System State Estimation and Observability",
    26: "Distribution-System Protection and Reconfiguration",
    27: "Smart-Meter Analytics and Load Profiling",
    28: "Energy Technology Innovation and Deployment",
    29: "Building Energy Flexibility and Demand Response",
    30: "Power-System Review and Synthesis Studies",
})

scopus_topic_labels.update({
    21: "Distributed Optimization and ADMM",
    22: "Demand-Side Management and Demand Response",
    23: "Quantum Computing for Power-System Optimization",
    24: "Data-Driven Power-System Analysis and Security Assessment",
    25: "Robust, Stochastic, and Probabilistic Optimization",
    26: "Cybersecurity and False-Data Injection Attacks",
    27: "Electric-Vehicle Charging and Infrastructure Planning",
    28: "Economic Dispatch and Unit Scheduling",
    29: "Graph Neural Networks and Topology-Aware Learning",
    30: "Safe and Physics-Informed Reinforcement Learning",
})

In [52]:
show_topic_range(
    "overton",
    31,
    40,
)

OVERTON — Topic 31

Top terms:
market, electr, price, particip, mechan, trade, competit, paper, consum, prosum

Representative publications:

1. Competitive electricity markets with consumer subscription service in a smart grid
   Topic probability: 0.3182

2. Bid Forwarding as a Way to Connect Sequential Markets: Opportunities and Barriers
   Topic probability: 0.3061

3. Wind energy aggregation: A coalitional game approach
   Topic probability: 0.2857

4. Smart contract-based campus demonstration of decentralized transactive energy auctions
   Topic probability: 0.2792

5. Nash equilibria in electricity markets with discrete prices
   Topic probability: 0.2765


OVERTON — Topic 32

Top terms:
frequenc, system, stabil, respons, dynam, generat, island, transient, mode, synchron

Representative publications:

1. Stability Evaluation of AC/DC Hybrid Microgrids Considering Bidirectional Power Flow through the Interlinking Converters
   Topic probability: 0.2671

2. Coherent swing instabil

In [53]:
show_topic_range(
    "scopus",
    31,
    40,
)

SCOPUS — Topic 31

Top terms:
distribut, network, der, resourc, node, activ, ieee, feeder, reconfigur, radial

Representative publications:

1. Quantifying the Impact of Day-ahead Renewable Forecasts on DER Hosting Capacity Estimation
   Topic probability: 0.2071

2. Analysis on the Reactive Power Regulation of Inverter and Voltage in Distribution Network
   Topic probability: 0.1975

3. Distributed control of active distribution networks to support voltage control in subtransmission networks
   Topic probability: 0.1925

4. Network reconfiguration at the power distribution system with dispersed generations for loss reduction
   Topic probability: 0.1866

5. Artificial Intelligence for Hosting Capacity Analysis: A Systematic Literature Review
   Topic probability: 0.1705


SCOPUS — Topic 32

Top terms:
problem, solut, solv, opf, optim, constraint, program, formul, feasibl, linear

Representative publications:

1. A non-convex alternating direction method of multipliers heuristic for op

In [54]:
overton_topic_labels.update({
    31: "Electricity Markets and Transactive Energy",
    32: "Frequency Stability and Converter-Dominated Grid Dynamics",
    33: "Wind-Turbine Wakes and Wind-Farm Aerodynamics",
    34: "Probabilistic Renewable-Energy Forecasting",
    35: "Solar Irradiance Measurement and Resource Assessment",
    36: "Power-System Computational and Assessment Methods",
    37: "Energy Transition, Communities, and Energy Justice",
    38: "Energy Materials, Batteries, and Electrochemical Technologies",
    39: "Dynamic Line Rating and Transmission Capacity",
    40: "Microgrid and Converter Control Strategies",
})

scopus_topic_labels.update({
    31: "Active Distribution Networks and DER Integration",
    32: "Mathematical Programming and OPF Solution Methods",
    33: "Power-System Stability and Oscillation Control",
    34: "Power-System Planning and Reliability Studies",
    35: "Power Electronics and Converter Systems",
    36: "Transmission Security and Contingency Assessment",
    37: "Renewable-Energy Integration and Management",
    38: "Battery and Hybrid Energy Storage Systems",
    39: "Data-Driven Smart-Energy Methods",
    40: "Grid Resilience and Cascading-Failure Risk",
})

In [55]:
show_topic_range(
    "overton",
    41,
    50,
)

OVERTON — Topic 41

Top terms:
emiss, gas, fuel, carbon, reduc, environment, reduct, fossil, product, decarbon

Representative publications:

1. A spatial and fleet disaggregated approach to calculating the NOX emissions inventory for non-road mobile machinery in London
   Topic probability: 0.4286

2. Decarbonizing maritime transport: The importance of engine technology and regulations for LNG to serve as a transition fuel
   Topic probability: 0.4175

3. Environmental sustainability of cooking fuels in remote communities: Life cycle and local impacts
   Topic probability: 0.3233

4. Total Methane and CO2Emissions from Liquefied Natural Gas Carrier Ships: The First Primary Measurements
   Topic probability: 0.3214

5. Sustainability implications of different carbon dioxide removal technologies in the context of Europe's climate neutrality goal
   Topic probability: 0.3085


OVERTON — Topic 42

Top terms:
energi, renew, sourc, resourc, increas, share, elsevi, sustain, res, effici

Repr

In [56]:
show_topic_range(
    "scopus",
    41,
    50,
)

SCOPUS — Topic 41

Top terms:
fault, protect, detect, classif, current, scheme, featur, transform, accuraci, time

Representative publications:

1. Component identification and defect detection in transmission lines based on deep learning
   Topic probability: 0.2978

2. Fault location of distribution network with distributed generation based on Karrenbauer transform and support vector machine regression
   Topic probability: 0.2865

3. SVM and DWT based Detection and Classification of Microgrid Faults using Single Point Measurement
   Topic probability: 0.2857

4. Decentralized cooperative protection strategy for smart distribution grid using Multi-Agent System
   Topic probability: 0.2848

5. Detection and Classification of Faults Using Intelligent Approach in Microgrid Network
   Topic probability: 0.2733


SCOPUS — Topic 42

Top terms:
algorithm, problem, search, converg, optim, test, metaheurist, hybrid, compar, solv

Representative publications:

1. An enhanced arithmetic optimiz

In [57]:
overton_topic_labels.update({
    41: "Carbon Emissions, Fuels, and Decarbonization",
    42: "Renewable-Energy Technologies and Policy",
    43: "Energy-System Costs and Economic Assessment",
    44: "Machine Learning for Power-System Monitoring and Assessment",
    45: "Power Electronics and Converter Technologies",
    46: "Future Grid Technologies and Electricity Demand",
    47: "Grid Resilience and Extreme-Event Risk",
    48: "Solar-PV Performance, Soiling, and Maintenance",
    49: "Grid Technology and System Integration Methods",
    50: "Energy Data Analytics and Anomaly Detection",
})

scopus_topic_labels.update({
    41: "Fault Detection, Classification, and Protection",
    42: "Metaheuristic Optimization Algorithms",
    43: "Power-System Engineering and Technology Applications",
    44: "Power-Flow and Network Calculation Methods",
    45: "Photovoltaic MPPT and Adaptive Control",
    46: "Load-Frequency Control and Frequency Regulation",
    47: "Wind-Power Modeling and Prediction",
    48: "Hybrid Computational Optimization Methods",
    49: "Power-System Data Analytics and Synthetic Data",
    50: "Bibliometric and Systematic Review Studies",
})

In [58]:
print(
    "Overton labels:",
    len(overton_topic_labels)
)

print(
    "Scopus labels :",
    len(scopus_topic_labels)
)

print(
    "Missing Overton:",
    sorted(
        set(range(1, 51))
        - set(overton_topic_labels)
    )
)

print(
    "Missing Scopus :",
    sorted(
        set(range(1, 51))
        - set(scopus_topic_labels)
    )
)

Overton labels: 50
Scopus labels : 50
Missing Overton: []
Missing Scopus : []


In [59]:
topic_label_maps = {
    "overton": overton_topic_labels,
    "scopus": scopus_topic_labels,
}

for corpus_name in [
    "overton",
    "scopus",
]:

    interpretation = (
        topic_interpretation_tables[
            corpus_name
        ]
        .copy()
    )

    interpretation[
        "Topic_Label"
    ] = (
        interpretation["Topic"]
        .map(
            topic_label_maps[
                corpus_name
            ]
        )
    )

    topic_interpretation_tables[
        corpus_name
    ] = interpretation

    print()
    print(corpus_name.upper())
    print("=" * len(corpus_name))

    display(
        interpretation[
            [
                "Topic",
                "Topic_Label",
                "Top_Terms",
            ]
        ]
    )


OVERTON


,Topic,Topic_Label,Top_Terms
0,1,Power-System Flexibility and Grid Planning,"oper, system, flexibl, plan, integr, consid, c..."
1,2,Offshore Wind and Marine Energy Modeling,"test, condit, result, use, wave, perform, expe..."
2,3,Energy and Environmental Impact Factors,"effect, factor, rate, increas, depend, also, m..."
3,4,Electric Machine and Wind Generator Design,"design, effici, perform, can, improv, hybrid, ..."
4,5,Power-System Modeling Standards and Software T...,"applic, develop, paper, standard, includ, new,..."
5,6,Energy-System Concepts and Technology Assessment,"term, main, author, relat, concept, take, one,..."
6,7,Computational Frameworks and Algorithmic Methods,"approach, framework, can, present, process, st..."
7,8,Electricity-Sector Policy and Market Reform,"polici, electr, countri, sector, industri, sup..."
8,9,Energy Research Infrastructure and Nuclear/Fus...,"develop, project, engin, research, nation, rig..."
9,10,Operating Reserves and Renewable Balancing,"generat, reserv, unit, variabl, penetr, dispat..."



SCOPUS


,Topic,Topic_Label,Top_Terms
0,1,Neural Networks and Physics-Informed Learning,"network, neural, train, use, predict, deep, ar..."
1,2,Voltage and Reactive Power Control,"voltag, loss, reactiv, activ, bus, ieee, regul..."
2,3,Power-System State Estimation,"estim, state, measur, paramet, method, use, ac..."
3,4,Electricity Markets and Peer-to-Peer Energy Tr...,"market, energi, price, electr, trade, mechan, ..."
4,5,"Grid Reliability, Risk, and Stability Challenges","can, will, due, issu, one, also, howev, may, m..."
5,6,Data-Driven and Linear Power-Flow Modeling,"model, develop, propos, simul, linear, datadri..."
6,7,Microgrid Operation and Energy Management,"microgrid, oper, manag, system, generat, mode,..."
7,8,Power-System Engineering Applications,"system, reserv, ltd, engin, right, publish, te..."
8,9,Digitalization and Intelligent Energy Systems,"technolog, energi, intellig, sustain, artifici..."
9,10,Metaheuristic Optimization Methods,"techniqu, use, differ, perform, various, sugge..."


In [60]:
for corpus_name, interpretation in (
    topic_interpretation_tables.items()
):

    output_file = (
        INTERPRETATION_DIR
        / f"{corpus_name}_topic_interpretation_labeled.csv"
    )

    interpretation.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():8s}: "
        f"{output_file}"
    )

OVERTON : /nfs/mfirdausi/project/review_paper_2/output/topic_interpretation/overton_topic_interpretation_labeled.csv
SCOPUS  : /nfs/mfirdausi/project/review_paper_2/output/topic_interpretation/scopus_topic_interpretation_labeled.csv


### 7.2 Topic Prevalence

Topic prevalence is calculated from the complete document-topic
probability distribution. For topic \(k\), prevalence is defined as
the mean topic probability across all documents in the corresponding
corpus. This probability-weighted measure preserves the mixed-membership
structure of LDA rather than assigning each publication exclusively to
its dominant topic.

In [61]:
topic_prevalence_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    prevalence = (
        final_lda[
            corpus_name
        ]["prevalence"]
        .copy()
    )

    # Add human-readable topic labels.
    prevalence[
        "Topic_Label"
    ] = (
        prevalence["Topic"]
        .map(
            topic_label_maps[
                corpus_name
            ]
        )
    )

    # Convert mean probability to percentage.
    prevalence[
        "Prevalence_Percent"
    ] = (
        prevalence[
            "Mean_Probability"
        ]
        * 100
    )

    prevalence = (
        prevalence
        .sort_values(
            "Prevalence_Percent",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    prevalence[
        "Rank"
    ] = (
        prevalence.index
        + 1
    )

    prevalence = prevalence[
        [
            "Rank",
            "Topic",
            "Topic_Label",
            "Mean_Probability",
            "Prevalence_Percent",
        ]
    ]

    topic_prevalence_tables[
        corpus_name
    ] = prevalence

    print()
    print(corpus_name.upper())
    print("=" * len(corpus_name))

    print(
        "Total prevalence:",
        f"{prevalence['Prevalence_Percent'].sum():.6f}%"
    )

    display(
        prevalence.head(15)
    )


OVERTON
Total prevalence: 100.000000%


,Rank,Topic,Topic_Label,Mean_Probability,Prevalence_Percent
0,1,22,Convex Relaxations and Mathematical Programmin...,0.022991,2.299056
1,2,26,Distribution-System Protection and Reconfigura...,0.022732,2.273240
2,3,33,Wind-Turbine Wakes and Wind-Farm Aerodynamics,0.022186,2.218621
3,4,30,Power-System Review and Synthesis Studies,0.022112,2.211217
4,5,35,Solar Irradiance Measurement and Resource Asse...,0.021932,2.193161
5,6,45,Power Electronics and Converter Technologies,0.021335,2.133460
6,7,42,Renewable-Energy Technologies and Policy,0.021218,2.121834
7,8,11,Smart-Grid Communications and Cybersecurity,0.021180,2.117991
8,9,25,Power-System State Estimation and Observability,0.021153,2.115275
9,10,17,Power-System Modeling and Simulation,0.020848,2.084812



SCOPUS
Total prevalence: 100.000000%


,Rank,Topic,Topic_Label,Mean_Probability,Prevalence_Percent
0,1,50,Bibliometric and Systematic Review Studies,0.024941,2.494135
1,2,32,Mathematical Programming and OPF Solution Methods,0.024178,2.417817
2,3,9,Digitalization and Intelligent Energy Systems,0.022733,2.273343
3,4,42,Metaheuristic Optimization Algorithms,0.022492,2.249235
4,5,37,Renewable-Energy Integration and Management,0.021736,2.173618
5,6,39,Data-Driven Smart-Energy Methods,0.021723,2.172327
6,7,21,Distributed Optimization and ADMM,0.021484,2.148377
7,8,16,Energy and Power Forecasting,0.021466,2.146579
8,9,18,Multi-Objective and Evolutionary Optimization,0.021435,2.143512
9,10,5,"Grid Reliability, Risk, and Stability Challenges",0.021154,2.115357


In [62]:
PREVALENCE_DIR = (
    OUTPUT_DIR
    / "topic_prevalence"
)

PREVALENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for corpus_name, prevalence in (
    topic_prevalence_tables.items()
):

    output_file = (
        PREVALENCE_DIR
        / f"{corpus_name}_topic_prevalence.csv"
    )

    prevalence.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():8s}: "
        f"{output_file}"
    )

OVERTON : /nfs/mfirdausi/project/review_paper_2/output/topic_prevalence/overton_topic_prevalence.csv
SCOPUS  : /nfs/mfirdausi/project/review_paper_2/output/topic_prevalence/scopus_topic_prevalence.csv


In [63]:
for corpus_name, prevalence in (
    topic_prevalence_tables.items()
):

    print()
    print(corpus_name.upper())
    print("-" * len(corpus_name))

    for n in [
        5,
        10,
        20,
    ]:

        cumulative_share = (
            prevalence[
                "Prevalence_Percent"
            ]
            .head(n)
            .sum()
        )

        print(
            f"Top {n:2d} topics : "
            f"{cumulative_share:.2f}%"
        )

    print(
        "\nLargest topic  :",
        f"{prevalence['Prevalence_Percent'].max():.2f}%"
    )

    print(
        "Smallest topic :",
        f"{prevalence['Prevalence_Percent'].min():.2f}%"
    )

    print(
        "Mean topic     :",
        f"{prevalence['Prevalence_Percent'].mean():.2f}%"
    )

    print(
        "Median topic   :",
        f"{prevalence['Prevalence_Percent'].median():.2f}%"
    )


OVERTON
-------
Top  5 topics : 11.20%
Top 10 topics : 21.77%
Top 20 topics : 42.22%

Largest topic  : 2.30%
Smallest topic : 1.83%
Mean topic     : 2.00%
Median topic   : 1.97%

SCOPUS
------
Top  5 topics : 11.61%
Top 10 topics : 22.33%
Top 20 topics : 42.58%

Largest topic  : 2.49%
Smallest topic : 1.79%
Mean topic     : 2.00%
Median topic   : 1.97%


In [64]:
overton_top10 = (
    topic_prevalence_tables[
        "overton"
    ]
    .head(10)
    [
        [
            "Rank",
            "Topic",
            "Topic_Label",
            "Prevalence_Percent",
        ]
    ]
    .copy()
)

scopus_top10 = (
    topic_prevalence_tables[
        "scopus"
    ]
    .head(10)
    [
        [
            "Rank",
            "Topic",
            "Topic_Label",
            "Prevalence_Percent",
        ]
    ]
    .copy()
)

print("OVERTON — Top 10 Topics")
display(
    overton_top10
)

print("SCOPUS — Top 10 Topics")
display(
    scopus_top10
)

OVERTON — Top 10 Topics


,Rank,Topic,Topic_Label,Prevalence_Percent
0,1,22,Convex Relaxations and Mathematical Programmin...,2.299056
1,2,26,Distribution-System Protection and Reconfigura...,2.273240
2,3,33,Wind-Turbine Wakes and Wind-Farm Aerodynamics,2.218621
3,4,30,Power-System Review and Synthesis Studies,2.211217
4,5,35,Solar Irradiance Measurement and Resource Asse...,2.193161
5,6,45,Power Electronics and Converter Technologies,2.133460
6,7,42,Renewable-Energy Technologies and Policy,2.121834
7,8,11,Smart-Grid Communications and Cybersecurity,2.117991
8,9,25,Power-System State Estimation and Observability,2.115275
9,10,17,Power-System Modeling and Simulation,2.084812


SCOPUS — Top 10 Topics


,Rank,Topic,Topic_Label,Prevalence_Percent
0,1,50,Bibliometric and Systematic Review Studies,2.494135
1,2,32,Mathematical Programming and OPF Solution Methods,2.417817
2,3,9,Digitalization and Intelligent Energy Systems,2.273343
3,4,42,Metaheuristic Optimization Algorithms,2.249235
4,5,37,Renewable-Energy Integration and Management,2.173618
5,6,39,Data-Driven Smart-Energy Methods,2.172327
6,7,21,Distributed Optimization and ADMM,2.148377
7,8,16,Energy and Power Forecasting,2.146579
8,9,18,Multi-Objective and Evolutionary Optimization,2.143512
9,10,5,"Grid Reliability, Risk, and Stability Challenges",2.115357


### 7.3 Cross-Corpus Meta-Theme Mapping

Because the Overton and Scopus LDA models were estimated independently,
topic identifiers and thematic boundaries are not directly comparable
across corpora. The 50 latent topics from each corpus are therefore
mapped to a common set of higher-level research meta-themes based on
their interpreted topic labels, top terms, and representative
publications.

The latent-topic probabilities are subsequently aggregated within each
meta-theme, preserving the complete mixed-membership topic
distributions while enabling direct comparison between the
policy-facing and academic corpora.

In [70]:
META_THEMES = [
    "Optimization and Computational Methods",
    "Machine Learning and Data-Driven Methods",
    "Power-System Operation, Stability, and Reliability",
    "Distribution Systems, Microgrids, and Smart Grids",
    "Renewable Energy and Resource Integration",
    "Energy Storage and Electric Vehicles",
    "Markets, Flexibility, and Demand Response",
    "Power Electronics and Grid Technologies",
    "Climate, Environment, and Decarbonization",
    "Policy, Society, and Energy Transition",
    "Cross-Cutting and Review Studies",
]

print(
    "Number of common meta-themes:",
    len(META_THEMES)
)

for i, theme in enumerate(
    META_THEMES,
    start=1,
):
    print(
        f"{i:2d}. {theme}"
    )

Number of common meta-themes: 11
 1. Optimization and Computational Methods
 2. Machine Learning and Data-Driven Methods
 3. Power-System Operation, Stability, and Reliability
 4. Distribution Systems, Microgrids, and Smart Grids
 5. Renewable Energy and Resource Integration
 6. Energy Storage and Electric Vehicles
 7. Markets, Flexibility, and Demand Response
 8. Power Electronics and Grid Technologies
 9. Climate, Environment, and Decarbonization
10. Policy, Society, and Energy Transition
11. Cross-Cutting and Review Studies


In [71]:
overton_meta_theme_map = {

    # -----------------------------------------------------
    # Optimization and computational methods
    # -----------------------------------------------------
    7: "Optimization and Computational Methods",
    22: "Optimization and Computational Methods",
    36: "Optimization and Computational Methods",

    # -----------------------------------------------------
    # Machine learning and data-driven methods
    # -----------------------------------------------------
    27: "Machine Learning and Data-Driven Methods",
    44: "Machine Learning and Data-Driven Methods",
    50: "Machine Learning and Data-Driven Methods",

    # -----------------------------------------------------
    # Operation, stability and reliability
    # -----------------------------------------------------
    10: "Power-System Operation, Stability, and Reliability",
    17: "Power-System Operation, Stability, and Reliability",
    21: "Power-System Operation, Stability, and Reliability",
    25: "Power-System Operation, Stability, and Reliability",
    32: "Power-System Operation, Stability, and Reliability",
    47: "Power-System Operation, Stability, and Reliability",

    # -----------------------------------------------------
    # Distribution, microgrids and smart grids
    # -----------------------------------------------------
    5: "Distribution Systems, Microgrids, and Smart Grids",
    11: "Distribution Systems, Microgrids, and Smart Grids",
    20: "Distribution Systems, Microgrids, and Smart Grids",
    26: "Distribution Systems, Microgrids, and Smart Grids",

    # -----------------------------------------------------
    # Renewable energy and resource integration
    # -----------------------------------------------------
    2: "Renewable Energy and Resource Integration",
    16: "Renewable Energy and Resource Integration",
    23: "Renewable Energy and Resource Integration",
    33: "Renewable Energy and Resource Integration",
    34: "Renewable Energy and Resource Integration",
    35: "Renewable Energy and Resource Integration",
    42: "Renewable Energy and Resource Integration",
    48: "Renewable Energy and Resource Integration",

    # -----------------------------------------------------
    # Storage and electric vehicles
    # -----------------------------------------------------
    12: "Energy Storage and Electric Vehicles",
    24: "Energy Storage and Electric Vehicles",
    38: "Energy Storage and Electric Vehicles",

    # -----------------------------------------------------
    # Markets, flexibility and demand response
    # -----------------------------------------------------
    1: "Markets, Flexibility, and Demand Response",
    13: "Markets, Flexibility, and Demand Response",
    29: "Markets, Flexibility, and Demand Response",
    31: "Markets, Flexibility, and Demand Response",
    43: "Markets, Flexibility, and Demand Response",

    # -----------------------------------------------------
    # Power electronics and grid technologies
    # -----------------------------------------------------
    4: "Power Electronics and Grid Technologies",
    39: "Power Electronics and Grid Technologies",
    40: "Power Electronics and Grid Technologies",
    45: "Power Electronics and Grid Technologies",
    49: "Power Electronics and Grid Technologies",

    # -----------------------------------------------------
    # Climate/environment/decarbonization
    # -----------------------------------------------------
    3: "Climate, Environment, and Decarbonization",
    15: "Climate, Environment, and Decarbonization",
    18: "Climate, Environment, and Decarbonization",
    41: "Climate, Environment, and Decarbonization",

    # -----------------------------------------------------
    # Policy, society and transition
    # -----------------------------------------------------
    8: "Policy, Society, and Energy Transition",
    9: "Policy, Society, and Energy Transition",
    19: "Policy, Society, and Energy Transition",
    28: "Policy, Society, and Energy Transition",
    37: "Policy, Society, and Energy Transition",
    46: "Policy, Society, and Energy Transition",

    # -----------------------------------------------------
    # Cross-cutting/reviews
    # -----------------------------------------------------
    6: "Cross-Cutting and Review Studies",
    14: "Cross-Cutting and Review Studies",
    30: "Cross-Cutting and Review Studies",
}

In [76]:
scopus_meta_theme_map = {

    # -----------------------------------------------------
    # Optimization and computational methods
    # -----------------------------------------------------
    10: "Optimization and Computational Methods",
    12: "Optimization and Computational Methods",
    13: "Optimization and Computational Methods",
    18: "Optimization and Computational Methods",
    20: "Optimization and Computational Methods",
    21: "Optimization and Computational Methods",
    23: "Optimization and Computational Methods",
    25: "Optimization and Computational Methods",
    28: "Optimization and Computational Methods",
    32: "Optimization and Computational Methods",
    42: "Optimization and Computational Methods",
    44: "Optimization and Computational Methods",
    48: "Optimization and Computational Methods",

    # -----------------------------------------------------
    # Machine learning and data-driven methods
    # -----------------------------------------------------
    1: "Machine Learning and Data-Driven Methods",
    3: "Machine Learning and Data-Driven Methods",
    6: "Machine Learning and Data-Driven Methods",
    16: "Machine Learning and Data-Driven Methods",
    24: "Machine Learning and Data-Driven Methods",
    29: "Machine Learning and Data-Driven Methods",
    30: "Machine Learning and Data-Driven Methods",
    39: "Machine Learning and Data-Driven Methods",
    49: "Machine Learning and Data-Driven Methods",

    # -----------------------------------------------------
    # Operation, stability and reliability
    # -----------------------------------------------------
    2: "Power-System Operation, Stability, and Reliability",
    5: "Power-System Operation, Stability, and Reliability",
    33: "Power-System Operation, Stability, and Reliability",
    36: "Power-System Operation, Stability, and Reliability",
    40: "Power-System Operation, Stability, and Reliability",
    41: "Power-System Operation, Stability, and Reliability",
    46: "Power-System Operation, Stability, and Reliability",
    26: "Power-System Operation, Stability, and Reliability",

    # -----------------------------------------------------
    # Distribution, microgrids and smart grids
    # -----------------------------------------------------
    7: "Distribution Systems, Microgrids, and Smart Grids",
    17: "Distribution Systems, Microgrids, and Smart Grids",
    31: "Distribution Systems, Microgrids, and Smart Grids",

    # -----------------------------------------------------
    # Renewable energy and resource integration
    # -----------------------------------------------------
    15: "Renewable Energy and Resource Integration",
    37: "Renewable Energy and Resource Integration",
    45: "Renewable Energy and Resource Integration",
    47: "Renewable Energy and Resource Integration",

    # -----------------------------------------------------
    # Storage and EV
    # -----------------------------------------------------
    27: "Energy Storage and Electric Vehicles",
    38: "Energy Storage and Electric Vehicles",

    # -----------------------------------------------------
    # Markets/flexibility/demand response
    # -----------------------------------------------------
    4: "Markets, Flexibility, and Demand Response",
    11: "Markets, Flexibility, and Demand Response",
    22: "Markets, Flexibility, and Demand Response",

    # -----------------------------------------------------
    # Power electronics/grid technologies
    # -----------------------------------------------------
    35: "Power Electronics and Grid Technologies",

    # -----------------------------------------------------
    # Climate/environment/decarbonization
    # -----------------------------------------------------
    19: "Climate, Environment, and Decarbonization",

    # -----------------------------------------------------
    # Policy/society/transition
    # -----------------------------------------------------
    9: "Policy, Society, and Energy Transition",

    # -----------------------------------------------------
    # Cross-cutting/reviews
    # -----------------------------------------------------
    8: "Cross-Cutting and Review Studies",
    14: "Cross-Cutting and Review Studies",
    34: "Cross-Cutting and Review Studies",
    43: "Cross-Cutting and Review Studies",
    50: "Cross-Cutting and Review Studies",
}

In [77]:
meta_theme_maps = {
    "overton": overton_meta_theme_map,
    "scopus": scopus_meta_theme_map,
}

for corpus_name, mapping in (
    meta_theme_maps.items()
):

    expected_topics = set(
        range(1, 51)
    )

    mapped_topics = set(
        mapping.keys()
    )

    missing = sorted(
        expected_topics
        - mapped_topics
    )

    extra = sorted(
        mapped_topics
        - expected_topics
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        "Mapped topics :",
        len(mapped_topics)
    )

    print(
        "Missing       :",
        missing
    )

    print(
        "Extra         :",
        extra
    )

    print()

OVERTON
-------
Mapped topics : 50
Missing       : []
Extra         : []

SCOPUS
------
Mapped topics : 50
Missing       : []
Extra         : []



In [78]:
# ---------------------------------------------------------
# Create mapping tables
# ---------------------------------------------------------

meta_theme_mapping_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    mapping_df = (
        topic_interpretation_tables[
            corpus_name
        ][
            [
                "Topic",
                "Topic_Label",
            ]
        ]
        .copy()
    )

    mapping_df[
        "Meta_Theme"
    ] = (
        mapping_df["Topic"]
        .map(
            meta_theme_maps[
                corpus_name
            ]
        )
    )

    mapping_df = (
        mapping_df
        .sort_values(
            [
                "Meta_Theme",
                "Topic",
            ]
        )
        .reset_index(drop=True)
    )

    meta_theme_mapping_tables[
        corpus_name
    ] = mapping_df


# ---------------------------------------------------------
# Save mappings
# ---------------------------------------------------------

META_THEME_DIR = (
    OUTPUT_DIR
    / "meta_theme_analysis"
)

META_THEME_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for corpus_name, mapping_df in (
    meta_theme_mapping_tables.items()
):

    output_file = (
        META_THEME_DIR
        / f"{corpus_name}_topic_to_meta_theme.csv"
    )

    mapping_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8",
    )

    print(
        f"{corpus_name.upper():8s}: "
        f"{len(mapping_df)} topics saved"
    )

    print(
        "  ",
        output_file
    )

OVERTON : 50 topics saved
   /nfs/mfirdausi/project/review_paper_2/output/meta_theme_analysis/overton_topic_to_meta_theme.csv
SCOPUS  : 50 topics saved
   /nfs/mfirdausi/project/review_paper_2/output/meta_theme_analysis/scopus_topic_to_meta_theme.csv


In [79]:
for corpus_name in [
    "overton",
    "scopus",
]:

    saved_file = (
        META_THEME_DIR
        / f"{corpus_name}_topic_to_meta_theme.csv"
    )

    check_df = pd.read_csv(
        saved_file
    )

    print(
        f"{corpus_name.upper():8s} | "
        f"Rows: {len(check_df)} | "
        f"Missing labels: "
        f"{check_df['Topic_Label'].isna().sum()} | "
        f"Missing meta-themes: "
        f"{check_df['Meta_Theme'].isna().sum()}"
    )

OVERTON  | Rows: 50 | Missing labels: 0 | Missing meta-themes: 0
SCOPUS   | Rows: 50 | Missing labels: 0 | Missing meta-themes: 0


In [80]:
meta_theme_prevalence = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    # Topic-level prevalence
    prevalence = (
        topic_prevalence_tables[
            corpus_name
        ][
            [
                "Topic",
                "Topic_Label",
                "Prevalence_Percent",
            ]
        ]
        .copy()
    )

    # Topic -> meta-theme mapping
    mapping = (
        meta_theme_mapping_tables[
            corpus_name
        ][
            [
                "Topic",
                "Meta_Theme",
            ]
        ]
        .copy()
    )

    # -----------------------------------------------------
    # Merge
    # -----------------------------------------------------

    merged = (
        prevalence
        .merge(
            mapping,
            on="Topic",
            how="left",
            validate="one_to_one",
        )
    )

    if merged["Meta_Theme"].isna().any():
        raise ValueError(
            f"Missing meta-theme assignments "
            f"for {corpus_name}"
        )

    # -----------------------------------------------------
    # Aggregate prevalence
    # -----------------------------------------------------

    aggregated = (
        merged
        .groupby(
            "Meta_Theme",
            as_index=False,
        )["Prevalence_Percent"]
        .sum()
        .rename(
            columns={
                "Prevalence_Percent":
                "Meta_Theme_Prevalence"
            }
        )
        .sort_values(
            "Meta_Theme_Prevalence",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    aggregated["Rank"] = (
        aggregated.index + 1
    )

    meta_theme_prevalence[
        corpus_name
    ] = aggregated

    print()
    print(corpus_name.upper())
    print("=" * len(corpus_name))

    print(
        "Total prevalence:",
        f"{aggregated['Meta_Theme_Prevalence'].sum():.6f}%"
    )

    display(
        aggregated[
            [
                "Rank",
                "Meta_Theme",
                "Meta_Theme_Prevalence",
            ]
        ]
    )


OVERTON
Total prevalence: 100.000000%


,Rank,Meta_Theme,Meta_Theme_Prevalence
0,1,Renewable Energy and Resource Integration,16.232708
1,2,"Power-System Operation, Stability, and Reliabi...",12.137651
2,3,"Policy, Society, and Energy Transition",11.877667
3,4,Power Electronics and Grid Technologies,9.795837
4,5,"Markets, Flexibility, and Demand Response",9.650724
5,6,"Distribution Systems, Microgrids, and Smart Grids",8.460937
6,7,"Climate, Environment, and Decarbonization",7.844959
7,8,Optimization and Computational Methods,6.267905
8,9,Cross-Cutting and Review Studies,6.015329
9,10,Machine Learning and Data-Driven Methods,5.910540



SCOPUS
Total prevalence: 100.000000%


,Rank,Meta_Theme,Meta_Theme_Prevalence
0,1,Optimization and Computational Methods,26.884539
1,2,Machine Learning and Data-Driven Methods,17.937459
2,3,"Power-System Operation, Stability, and Reliabi...",15.609353
3,4,Cross-Cutting and Review Studies,10.089870
4,5,Renewable Energy and Resource Integration,7.890067
5,6,"Distribution Systems, Microgrids, and Smart Grids",5.831549
6,7,"Markets, Flexibility, and Demand Response",5.771897
7,8,Energy Storage and Electric Vehicles,3.842298
8,9,"Policy, Society, and Energy Transition",2.273343
9,10,"Climate, Environment, and Decarbonization",1.943472


In [81]:
overton_meta = (
    meta_theme_prevalence[
        "overton"
    ][
        [
            "Meta_Theme",
            "Meta_Theme_Prevalence",
        ]
    ]
    .rename(
        columns={
            "Meta_Theme_Prevalence":
            "Overton_Percent"
        }
    )
)

scopus_meta = (
    meta_theme_prevalence[
        "scopus"
    ][
        [
            "Meta_Theme",
            "Meta_Theme_Prevalence",
        ]
    ]
    .rename(
        columns={
            "Meta_Theme_Prevalence":
            "Scopus_Percent"
        }
    )
)

meta_theme_comparison = (
    overton_meta
    .merge(
        scopus_meta,
        on="Meta_Theme",
        how="outer",
        validate="one_to_one",
    )
)

# Percentage-point difference:
# positive = relatively greater Overton emphasis
# negative = relatively greater Scopus emphasis
meta_theme_comparison[
    "Difference_pp"
] = (
    meta_theme_comparison[
        "Overton_Percent"
    ]
    -
    meta_theme_comparison[
        "Scopus_Percent"
    ]
)

meta_theme_comparison[
    "Absolute_Difference_pp"
] = (
    meta_theme_comparison[
        "Difference_pp"
    ]
    .abs()
)

meta_theme_comparison[
    "Relative_Emphasis"
] = (
    meta_theme_comparison[
        "Difference_pp"
    ]
    .apply(
        lambda x:
        "Overton"
        if x > 0
        else (
            "Scopus"
            if x < 0
            else "Equal"
        )
    )
)

meta_theme_comparison = (
    meta_theme_comparison
    .sort_values(
        "Absolute_Difference_pp",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    meta_theme_comparison[
        [
            "Meta_Theme",
            "Overton_Percent",
            "Scopus_Percent",
            "Difference_pp",
            "Relative_Emphasis",
        ]
    ]
)

,Meta_Theme,Overton_Percent,Scopus_Percent,Difference_pp,Relative_Emphasis
0,Optimization and Computational Methods,6.267905,26.884539,-20.616634,Scopus
1,Machine Learning and Data-Driven Methods,5.910540,17.937459,-12.026919,Scopus
2,"Policy, Society, and Energy Transition",11.877667,2.273343,9.604324,Overton
3,Renewable Energy and Resource Integration,16.232708,7.890067,8.342641,Overton
4,Power Electronics and Grid Technologies,9.795837,1.926153,7.869685,Overton
5,"Climate, Environment, and Decarbonization",7.844959,1.943472,5.901487,Overton
6,Cross-Cutting and Review Studies,6.015329,10.089870,-4.074540,Scopus
7,"Markets, Flexibility, and Demand Response",9.650724,5.771897,3.878828,Overton
8,"Power-System Operation, Stability, and Reliabi...",12.137651,15.609353,-3.471702,Scopus
9,"Distribution Systems, Microgrids, and Smart Grids",8.460937,5.831549,2.629388,Overton


In [82]:
print(
    "Overton total:",
    f"{meta_theme_comparison['Overton_Percent'].sum():.6f}%"
)

print(
    "Scopus total :",
    f"{meta_theme_comparison['Scopus_Percent'].sum():.6f}%"
)

print(
    "Number of meta-themes:",
    len(meta_theme_comparison)
)

Overton total: 100.000000%
Scopus total : 100.000000%
Number of meta-themes: 11


In [83]:
comparison_file = (
    META_THEME_DIR
    / "overton_scopus_meta_theme_comparison.csv"
)

meta_theme_comparison.to_csv(
    comparison_file,
    index=False,
    encoding="utf-8",
)

print(
    "Saved:",
    comparison_file
)

Saved: /nfs/mfirdausi/project/review_paper_2/output/meta_theme_analysis/overton_scopus_meta_theme_comparison.csv


### 7.4 Cross-Corpus Thematic Mapping


### 7.5 Policy-Facing vs Academic Research Priorities